# T cell analysis

Cleaned T-cell notebook for Soskic primary analyses plus Nathan and Cano-Gamez validation heatmaps. The notebook uses `scDRS-FM` naming throughout, restores the original Soskic UMAP / phenotype / pathway figures, and keeps the updated heatmap styling: square cells, cell-type counts in labels, all marginally associated cell types, and white independent-population annotations with black outlines.

For the Nature Genetics manuscript supplementary materials, each cell-type heatmap writes a tidy CSV in exact plotted order. Saved `indep_cells` assignments are limited to the plotted scDRS-FM traits and to trait × cell-type sections whose marginal × conditional proportion is **strictly greater than 5%**; exact 5% values are excluded, while placeholder signal labels such as `-1` are retained within passing sections.

## Update verification

This version retains the restored Soskic CD4 T-cell, independent-population, phenotype, method-comparison, and pathway analyses. It now saves UMAPs for every defined phenotype and disease trait, orders disease-score layers from highest to lowest score, uses smaller cell-type and independent-population markers, and omits significance stars from the final pathway heatmap.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from statsmodels.stats.multitest import multipletests

sns.set_context("notebook")
plt.rcParams["figure.dpi"] = 300

## Configuration

In [2]:
# === scDRS-FM reproduction: portable path anchor (injected, P5) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    # 1) explicit override wins
    env = _os.environ.get('SCDRSFM_BASE')
    if env:
        return _Path(env)
    # 2) search upward from CWD for the reproduction repo root
    #    (a directory containing both 'scDRS-FM-main' and 'scripts')
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'scDRS-FM-main').is_dir() and (d / 'scripts').is_dir():
            return d
    # 3) last resort: current working directory
    return here
BASE = _find_repo_root()
DATA = BASE / 'data'
RESULTS = BASE / 'results'
MAGMA_REF = BASE / 'magma_ref'
assert BASE.exists(), f'Repro root not found (set SCDRSFM_BASE to the repo root): {BASE}'


In [3]:
# -----------------------------------------------------------------------------
# Traits
# -----------------------------------------------------------------------------
ALL_TRAITS = [
    "PASS_CD_deLange2017",
    "PASS_Celiac",
    "PASS_IBD_deLange2017",
    "PASS_Lupus",
    "PASS_Multiple_sclerosis",
    "PASS_Primary_biliary_cirrhosis",
    "PASS_Rheumatoid_Arthritis",
    "PASS_Type_1_Diabetes",
    "PASS_UC_deLange2017",
    "UKB_460K.disease_AID_ALL",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP",
    "UKB_460K.disease_RESPIRATORY_ENT",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "UKB_460K.disease_ASTHMA_DIAGNOSED",

]

# Functional phenotype scores available in the Soskic scDRS results.
PHENOTYPE_TRAITS = [
    "Metallothionein", "Translation", "IL10-IL19", "OX40-EBI3", "CD172a-MERTK", "TIMD4-TIM3",
    "BCL2-FAM13A", "IEG", "SOX4-TOX2", "NME1-FABP5", "IEG3", "RGCC-MYADM", "Exhaustion", "ISG",
    "Cytotoxic", "CD40LG-TXNIP", "Mito", "HLA", "Heatshock", "IEG2", "Cytoskeleton", "CTLA4-CD38",
    "Multi-Cytokine", "ICOS-CD38",
]

TRAIT_LABELS = {
    "PASS_CD_deLange2017": "Crohn’s Disease (CD)",
    "PASS_UC_deLange2017": "Ulcerative Colitis (UC)",
    "PASS_IBD_deLange2017": "Inflammatory Bowel Disease (IBD)",
    "PASS_Celiac": "Celiac Disease (CeD)",
    "PASS_Lupus": "Lupus",
    "PASS_Multiple_sclerosis": "Multiple Sclerosis (MS)",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid Arthritis (RA)",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes (T1D)",
    "PASS_Primary_biliary_cirrhosis": "Primary Biliary Cholangitis (PBC)",
    "UKB_460K.disease_AID_ALL": "Autoimmune Traits (AIT)",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED": "Allergic Eczema (ECZ)",
    "UKB_460K.disease_RESPIRATORY_ENT": "Respiratory / ENT Disease (RESP-ENT)",
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP": "Hypothyroidism (HYPO)",
}

# Heatmap traits are grouped in the requested display order.
TRAIT_GROUPS = {
    "GI": [
    "PASS_CD_deLange2017",
    "PASS_UC_deLange2017",
    "PASS_IBD_deLange2017",
    "PASS_Celiac",
    ],
    "Sys.": [
    "PASS_Rheumatoid_Arthritis",
    "UKB_460K.disease_AID_ALL",
    ],
    "All.": [
    "UKB_460K.disease_ASTHMA_DIAGNOSED",
    "UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED",
    "UKB_460K.disease_RESPIRATORY_ENT",
    ],
    "Other": [
    "UKB_460K.disease_HYPOTHYROIDISM_SELF_REP",
    ],
}
TRAIT_GROUP_ORDER = ["GI", "Sys.", "All.", "Other"]
TRAIT_GROUP_COLORS = {
    "GI": "red",
    "Sys.": "blue",
    "All.": "green",
    "Other": "purple",
}
HEATMAP_TRAITS = [trait for group in TRAIT_GROUP_ORDER for trait in TRAIT_GROUPS[group]]

# Generate disease-score UMAPs for every trait considered in the notebook.
UMAP_TRAITS = list(ALL_TRAITS)

# IBD remains the focal trait for the downstream independent-population DE/pathway analysis.
UMAP_TRAIT = "PASS_IBD_deLange2017"

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
OUTPUT_DIR = Path("t_cell_analysis_outputs")
HEATMAP_DIR = OUTPUT_DIR / "heatmaps"
UMAP_DIR = OUTPUT_DIR / "umaps"
TABLE_DIR = OUTPUT_DIR / "tables"
for folder in [OUTPUT_DIR, HEATMAP_DIR, UMAP_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Nature Genetics manuscript supplementary tables generated by cell-type heatmaps.
MANUSCRIPT_SUPPLEMENTARY_DIR = Path("nature_genetics_manuscript_supplementary")
MANUSCRIPT_SUPPLEMENTARY_DIR.mkdir(parents=True, exist_ok=True)
HEATMAP_THRESHOLD = 0.05
SOSKIC_SCDRSFM_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "Soskic_scDRSFM_all_marginal_celltypes_cell_type_proportions.csv"
SOSKIC_SCDRS_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "Soskic_scDRS_celltype_assoc_heatmap_cell_type_proportions.csv"
SOSKIC_SCPAGWAS_CSV = MANUSCRIPT_SUPPLEMENTARY_DIR / "Soskic_scPagwas_celltype_assoc_heatmap_cell_type_proportions.csv"


def first_existing_path(*candidates: Path) -> Path:
    """Return the first candidate that exists, otherwise return the first candidate."""
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


SCDRSFM_SOSKIC_RESULTS = first_existing_path(
    RESULTS / "ct" / "soskic_immune_magic_ctrl",
)

SCDRS_SOSKIC_RESULTS = first_existing_path(
    RESULTS / "ct" / "soskic_immune_none_ctrl",
)

SCPAGWAS_SOSKIC_DIR = first_existing_path(
    RESULTS / "scpagwas" / "soskic",  # absent in this reproduction -> guarded
)


# Original Soskic phenotype-score results used for all functional phenotype UMAPs.
PHENO_SOSKIC_RESULTS = first_existing_path(
    RESULTS / "real" / "soskic_tcell",
)

# Functional T-cell phenotype gene sets used for the final pathway heatmap.
T_CELL_PHENO_DIR = first_existing_path(
    DATA / "gene_sets" / "t_cell_pheno",
)


@dataclass(frozen=True)
class DatasetConfig:
    name: str
    label: str
    h5ad_file: Path
    scdrsfm_results_dir: Path
    biocol: str
    fdr_alpha: float = 0.1
    marginal_metacell_col: str = "metacell"
    indep_sig_col: str = "independent_signal"
    apply_preprocessing: bool = True


DATASETS = {
    "soskic": DatasetConfig(
        name="soskic",
        label="Soskic",
        h5ad_file=first_existing_path(
            DATA / "subsets_10k" / "Soskic" / "soskic_100k.h5ad",
        ),
        scdrsfm_results_dir=SCDRSFM_SOSKIC_RESULTS,
        biocol="Cell_population",
        fdr_alpha=0.1,
    ),
    "nathan": DatasetConfig(
        name="nathan",
        label="Nathan",
        h5ad_file=DATA / "subsets_10k" / "Nathan" / "raw.h5ad",
        scdrsfm_results_dir=RESULTS / "ct" / "nathan_tcell_magic_ctrl",
        biocol="cluster_name",
        fdr_alpha=0.2,
    ),
    "canogamez": DatasetConfig(
        name="canogamez",
        label="Cano-Gamez",
        h5ad_file=DATA / "subsets_10k" / "Cano_Gamez" / "obj_raw.h5ad",
        scdrsfm_results_dir=RESULTS / "ct" / "canogamez_tcell_magic_ctrl",
        biocol="cluster.id",
        fdr_alpha=0.2,
    ),
}

## General utilities

In [4]:
def pick_first_existing_col(df: pd.DataFrame, candidates: Sequence[str], *, what: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"{what}: none of these columns exist: {tuple(candidates)}")


def bh_fdr_mask(pvals: Sequence[float], alpha: float = 0.05) -> np.ndarray:
    """Benjamini-Hochberg FDR reject mask, ignoring NaN/inf values."""
    p = np.asarray(pvals, dtype=float)
    ok = np.isfinite(p)
    out = np.zeros(len(p), dtype=bool)
    if ok.sum() == 0:
        return out
    reject, _, _, _ = multipletests(p[ok], alpha=alpha, method="fdr_bh")
    out[ok] = reject
    return out


def capfirst(value: str) -> str:
    value = str(value).strip()
    return value[:1].upper() + value[1:] if value else value


def parse_csv_index(value: object) -> pd.Index:
    if value is None:
        return pd.Index([], dtype=str)
    text = str(value)
    if not text or text == "nan":
        return pd.Index([], dtype=str)
    return pd.Index([x for x in text.split(",") if x], dtype=str)


def join_index(values: Iterable[object]) -> str:
    idx = pd.Index([str(x) for x in values]).unique()
    if len(idx) == 0:
        return ""
    try:
        idx = idx.sort_values()
    except Exception:
        pass
    return ",".join(idx.tolist())


def trait_group(trait: str) -> str:
    for group, traits in TRAIT_GROUPS.items():
        if trait in traits:
            return group
    return "Other"


def trait_color(trait: str) -> str:
    return TRAIT_GROUP_COLORS.get(trait_group(trait), "gray")


def color_ticklabels(ticklabels, colors, *, fontsize=24, rotation=None, ha=None):
    for tick, color in zip(ticklabels, colors):
        tick.set_color(color)
        tick.set_fontsize(fontsize)
        if rotation is not None:
            tick.set_rotation(rotation)
        if ha is not None:
            tick.set_ha(ha)


def add_right_group_lines(
    ax,
    color_counts: list[tuple[str, int]],
    names: list[str],
    *,
    x_axes: float = 1.01,
    text_offset: float = 0.02,
    linewidth: float = 4,
    fontsize: float = 32,
    fontweight: str = "bold",
):
    """Right-side group bars. color_counts must be in bottom-to-top y-axis order."""
    total = max(1, sum(count for _, count in color_counts))
    start = 0
    for (color, count), name in zip(color_counts, names):
        count = int(count)
        end = start + count
        ymin, ymax = start / total, end / total
        ax.add_line(
            Line2D(
                [x_axes, x_axes],
                [ymin, ymax],
                transform=ax.transAxes,
                color=color,
                linewidth=linewidth,
                solid_capstyle="butt",
                clip_on=False,
            )
        )
        if ymax > ymin:
            ax.text(
                x_axes + text_offset,
                (ymin + ymax) / 2,
                name,
                ha="left",
                va="center",
                fontsize=fontsize,
                color=color,
                fontweight=fontweight,
                transform=ax.transAxes,
                clip_on=False,
                rotation=-90,
            )
        start = end


def trait_group_segments_bottom_to_top(traits_top_to_bottom: list[str]) -> tuple[list[tuple[str, int]], list[str]]:
    """Return contiguous trait-group segments in bottom-to-top plotting order."""
    traits_bottom_to_top = list(reversed(traits_top_to_bottom))
    if not traits_bottom_to_top:
        return [], []

    names = [trait_group(traits_bottom_to_top[0])]
    counts = [0]
    for trait in traits_bottom_to_top:
        group = trait_group(trait)
        if group == names[-1]:
            counts[-1] += 1
        else:
            names.append(group)
            counts.append(1)
    color_counts = [(TRAIT_GROUP_COLORS.get(group, "gray"), count) for group, count in zip(names, counts)]
    return color_counts, names


def celltype_counts(adata: sc.AnnData, biocol: str) -> pd.Series:
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs missing {biocol!r}")
    return adata.obs[biocol].astype(str).value_counts()


def label_celltypes_with_counts(cell_types: Sequence[str], counts: pd.Series) -> list[str]:
    return [f"{capfirst(ct)} ({int(counts.get(ct, 0)):,})" for ct in cell_types]


def create_square_heatmap_figure(
    n_rows: int,
    n_cols: int,
    *,
    cell_size: float = 0.62,
    left_margin: float = 6.5,
    right_margin: float = 3.0,
    bottom_margin: float = 6.5,
    top_margin: float = 3.8,
):
    """Create an axis whose physical width/height is proportional to n_cols/n_rows."""
    heatmap_width = max(3.5, n_cols * cell_size)
    heatmap_height = max(3.5, n_rows * cell_size)
    fig_width = left_margin + heatmap_width + right_margin
    fig_height = bottom_margin + heatmap_height + top_margin

    fig = plt.figure(figsize=(fig_width, fig_height))
    ax_left = left_margin / fig_width
    ax_bottom = bottom_margin / fig_height
    ax_width = heatmap_width / fig_width
    ax_height = heatmap_height / fig_height
    ax = fig.add_axes([ax_left, ax_bottom, ax_width, ax_height])
    ax.set_aspect("equal", adjustable="box")
    return fig, ax, (ax_left, ax_bottom, ax_width, ax_height)


def load_and_preprocess_adata(config: DatasetConfig) -> sc.AnnData:
    adata = sc.read_h5ad(config.h5ad_file)
    print(f"{config.label}: loaded {config.h5ad_file}: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

    if config.apply_preprocessing:
        sc.pp.filter_cells(adata, min_genes=250)
        sc.pp.filter_genes(adata, min_cells=50)
        sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
        sc.pp.log1p(adata)
        print(f"{config.label}: after filtering: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

    if config.biocol not in adata.obs.columns:
        raise ValueError(f"{config.label}: adata.obs missing {config.biocol!r}")

    adata.obs_names = adata.obs_names.astype(str)
    return adata

## scDRS-FM result parsing

In [5]:
def cells_from_metacell_rows(df: pd.DataFrame, *, cell_ids_col: str = "cell_ids") -> pd.Index:
    if cell_ids_col not in df.columns:
        raise ValueError(f"Conditional file missing {cell_ids_col!r}")
    cells = pd.Index([], dtype=str)
    for value in df[cell_ids_col].astype(str).tolist():
        cells = cells.append(parse_csv_index(value))
    return pd.Index(cells.astype(str)).unique()


def format_independent_signal_value(value: object) -> object:
    """Return a clean signal label for output files, preserving -1 when present."""
    if pd.isna(value):
        return value

    try:
        value_float = float(value)
    except (TypeError, ValueError):
        return str(value)

    if np.isfinite(value_float) and value_float.is_integer():
        return int(value_float)
    return value_float




def _passes_heatmap_inclusion(value: object, threshold: float) -> bool:
    """Return True only for a finite proportion strictly above the heatmap cutoff."""
    try:
        value_float = float(value)
        threshold_float = float(threshold)
    except (TypeError, ValueError):
        return False
    return bool(np.isfinite(value_float) and value_float > threshold_float)

def build_marginal_x_conditional_signal_assignments(
    *,
    adata: sc.AnnData,
    marginal_sig_cells: pd.Index,
    df_cond_sig: pd.DataFrame,
    indep_sig_col: str,
) -> pd.DataFrame:
    """
    Build the per-trait independent-cell output table.

    Output columns match the expected downstream format:
      - marginal_x_conditional_cell_id
      - independent_signal

    The table is generated from conditionally significant metacells only, then
    intersected with marginally significant cells. It includes all non-null
    independent-signal labels in those rows, including -1 when the input uses
    -1 as a no-independent-signal label.
    """
    output_columns = ["marginal_x_conditional_cell_id", "independent_signal"]
    if len(df_cond_sig) == 0:
        return pd.DataFrame(columns=output_columns)

    if indep_sig_col not in df_cond_sig.columns:
        raise ValueError(f"Conditional dataframe missing {indep_sig_col!r}")

    numeric_signal_values = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce")
    numeric_signal_values = numeric_signal_values[numeric_signal_values.notna()]
    if len(numeric_signal_values) == 0:
        return pd.DataFrame(columns=output_columns)

    rows: list[pd.DataFrame] = []
    numeric_signal_all = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce")

    for signal_value in sorted(numeric_signal_values.unique()):
        signal_rows = df_cond_sig.loc[numeric_signal_all.eq(signal_value).fillna(False)]
        signal_cells = adata.obs_names.intersection(cells_from_metacell_rows(signal_rows))
        marginal_x_conditional_cells = marginal_sig_cells.intersection(signal_cells)

        if len(marginal_x_conditional_cells) == 0:
            continue

        cell_ids = pd.Index(marginal_x_conditional_cells.astype(str)).unique()
        try:
            cell_ids = cell_ids.sort_values()
        except Exception:
            pass

        rows.append(
            pd.DataFrame(
                {
                    "marginal_x_conditional_cell_id": cell_ids,
                    "independent_signal": format_independent_signal_value(signal_value),
                }
            )
        )

    if not rows:
        return pd.DataFrame(columns=output_columns)

    assignments = pd.concat(rows, ignore_index=True)
    assignments = assignments.drop_duplicates(output_columns)
    assignments = assignments.sort_values(
        ["independent_signal", "marginal_x_conditional_cell_id"],
        kind="mergesort",
    ).reset_index(drop=True)
    return assignments


def _filter_independent_cell_assignments_for_heatmap(
    *,
    adata: sc.AnnData,
    assignments: pd.DataFrame,
    heatmap_cell_ids: pd.Index,
    biocol: str,
    totals_by_type: pd.Series,
    threshold: float,
    allowed_cell_types: list[str] | None = None,
) -> pd.DataFrame:
    """
    Keep assignments only from trait × cell-type sections included in the heatmap.

    Inclusion is calculated from all marginal × conditional cells for the trait,
    using the same denominator and strict ``> threshold`` rule as the heatmap.
    Every assignment in a passing section is retained, including placeholder
    labels such as -1. ``allowed_cell_types`` can further restrict exports to
    the cell types that are actually present in a curated heatmap.
    """
    output_columns = ["marginal_x_conditional_cell_id", "independent_signal"]
    if assignments is None or len(assignments) == 0:
        return pd.DataFrame(columns=output_columns)

    missing_columns = [column for column in output_columns if column not in assignments.columns]
    if missing_columns:
        raise ValueError(f"Independent-cell assignments are missing columns: {missing_columns}")
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs is missing '{biocol}'")

    celltype_order = totals_by_type.index.astype(str).tolist()
    section_fractions = discovery_fraction_by_celltype(
        adata=adata,
        discovered_cell_ids=heatmap_cell_ids,
        biocol=biocol,
        celltype_order=celltype_order,
        totals_by_type=totals_by_type,
    )
    passing_cell_types = set(
        section_fractions.index[
            section_fractions.map(lambda value: _passes_heatmap_inclusion(value, threshold))
        ].astype(str)
    )
    if allowed_cell_types is not None:
        passing_cell_types.intersection_update(str(cell_type) for cell_type in allowed_cell_types)
    if not passing_cell_types:
        return pd.DataFrame(columns=output_columns)

    work = assignments.loc[:, output_columns].copy()
    work["marginal_x_conditional_cell_id"] = work[
        "marginal_x_conditional_cell_id"
    ].astype(str)

    valid_cell_ids = adata.obs_names.intersection(
        work["marginal_x_conditional_cell_id"].astype(str)
    )
    work = work.loc[
        work["marginal_x_conditional_cell_id"].isin(valid_cell_ids)
    ].copy()
    if len(work) == 0:
        return pd.DataFrame(columns=output_columns)

    cell_type_by_id = adata.obs[biocol].astype(str)
    assignment_cell_types = work["marginal_x_conditional_cell_id"].map(cell_type_by_id)
    work = work.loc[assignment_cell_types.isin(passing_cell_types), output_columns]
    work = work.drop_duplicates(output_columns)
    work = work.sort_values(
        ["independent_signal", "marginal_x_conditional_cell_id"],
        kind="mergesort",
    ).reset_index(drop=True)
    return work


def discovery_fraction_by_celltype(
    adata: sc.AnnData,
    discovered_cell_ids: pd.Index,
    *,
    biocol: str,
    celltype_order: list[str],
    totals_by_type: pd.Series,
) -> pd.Series:
    discovered_cell_ids = adata.obs_names.intersection(discovered_cell_ids.astype(str))
    if len(discovered_cell_ids) == 0:
        return pd.Series(0.0, index=celltype_order)

    discovered_counts = adata.obs.loc[discovered_cell_ids, biocol].astype(str).value_counts()
    frac = (discovered_counts / totals_by_type).reindex(celltype_order, fill_value=0.0)
    return frac.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)


def valid_signal_values(series: pd.Series) -> list[int]:
    values = pd.to_numeric(series, errors="coerce")
    values = values[values.notna() & (values >= 0)]
    return sorted(values.astype(int).unique().tolist())


def build_scdrsfm_celltype_tables(
    *,
    adata: sc.AnnData,
    results_dir: Path,
    traits: list[str],
    biocol: str,
    marginal_metacell_col: str = "metacell",
    fdr_alpha: float = 0.1,
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    indep_sig_col: str = "independent_signal",
    indep_cells_dir: Path | None = None,
    heatmap_threshold: float = 0.05,
    heatmap_traits: Sequence[str] | None = None,
    heatmap_cell_types: Sequence[str] | None = None,
    print_summaries: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Build the scDRS-FM heatmap inputs.

    Returns:
      df_marginal_props: fraction of marginal-significant cells per cell type.
      df_marginal_x_cond_props: fraction of marginal-significant cells that are also in
        conditional-significant metacells per cell type.
      df_signal_details: per-(trait, independent_signal) cell lists used for annotations.
      df_indep_signal_counts: per-trait summary counts.

    Side effect:
      If indep_cells_dir is provided, writes one gzip-compressed TSV per trait
      to indep_cells_dir / f"{trait}.gz" with columns
      marginal_x_conditional_cell_id and independent_signal.
    """
    results_dir = Path(results_dir)
    if indep_cells_dir is not None:
        indep_cells_dir = Path(indep_cells_dir)
        indep_cells_dir.mkdir(parents=True, exist_ok=True)
    heatmap_threshold = float(heatmap_threshold)
    if not np.isfinite(heatmap_threshold) or not 0 <= heatmap_threshold <= 1:
        raise ValueError("heatmap_threshold must be a finite value between 0 and 1.")
    heatmap_trait_set = None if heatmap_traits is None else {str(trait) for trait in heatmap_traits}
    counts = celltype_counts(adata, biocol)
    celltype_order = counts.index.tolist()

    rows_marginal: dict[str, pd.Series] = {}
    rows_intersect: dict[str, pd.Series] = {}
    signal_details_rows: dict[tuple[str, int], dict[str, object]] = {}
    signal_count_rows: dict[str, dict[str, int]] = {}

    for trait in traits:
        prefix = Path(trait).name
        marginal_file = results_dir / f"{prefix}.marginal_score.gz"
        conditional_file = results_dir / f"{prefix}.conditional.tagging_score.gz"

        if not marginal_file.exists():
            raise FileNotFoundError(f"Missing marginal score file: {marginal_file}")
        if not conditional_file.exists():
            raise FileNotFoundError(f"Missing conditional score file: {conditional_file}")

        df_marg = pd.read_csv(marginal_file, sep="\t", compression="infer", index_col=0)
        if marginal_metacell_col not in df_marg.columns:
            raise ValueError(f"{marginal_file} missing {marginal_metacell_col!r}")

        pcol_marg = pick_first_existing_col(df_marg, pval_col_candidates, what="marginal p-value")
        marg_sig_mask = bh_fdr_mask(df_marg[pcol_marg].to_numpy(), alpha=fdr_alpha)
        marg_sig_cells = adata.obs_names.intersection(df_marg.index[marg_sig_mask].astype(str))

        df_cond = pd.read_csv(conditional_file, sep="\t", compression="infer", index_col=0).copy()
        df_cond.index = pd.to_numeric(pd.Index(df_cond.index), errors="coerce")
        df_cond = df_cond.loc[df_cond.index.notna()]
        df_cond.index = df_cond.index.astype(int)

        if indep_sig_col not in df_cond.columns:
            raise ValueError(f"{conditional_file} missing {indep_sig_col!r}")
        if "cell_ids" not in df_cond.columns:
            raise ValueError(f"{conditional_file} missing 'cell_ids'")

        pcol_cond = pick_first_existing_col(df_cond, pval_col_candidates, what="conditional p-value")
        cond_sig_mask = bh_fdr_mask(df_cond[pcol_cond].to_numpy(), alpha=fdr_alpha)
        df_cond_sig = df_cond.loc[cond_sig_mask] if cond_sig_mask.any() else df_cond.iloc[0:0]

        cond_sig_cells = (
            adata.obs_names.intersection(cells_from_metacell_rows(df_cond_sig))
            if len(df_cond_sig) else pd.Index([], dtype=str)
        )
        marg_x_cond_cells = marg_sig_cells.intersection(cond_sig_cells)

        independent_cell_assignments_unfiltered = build_marginal_x_conditional_signal_assignments(
            adata=adata,
            marginal_sig_cells=marg_sig_cells,
            df_cond_sig=df_cond_sig,
            indep_sig_col=indep_sig_col,
        )
        if heatmap_trait_set is not None and str(trait) not in heatmap_trait_set:
            independent_cell_assignments = pd.DataFrame(
                columns=["marginal_x_conditional_cell_id", "independent_signal"]
            )
        else:
            independent_cell_assignments = _filter_independent_cell_assignments_for_heatmap(
                adata=adata,
                assignments=independent_cell_assignments_unfiltered,
                heatmap_cell_ids=marg_x_cond_cells,
                biocol=biocol,
                totals_by_type=counts,
                threshold=heatmap_threshold,
                allowed_cell_types=None if heatmap_cell_types is None else list(heatmap_cell_types),
            )
        if indep_cells_dir is not None:
            independent_cell_assignments.to_csv(
                indep_cells_dir / f"{prefix}.gz",
                sep="\t",
                index=False,
                compression="gzip",
            )

        rows_marginal[trait] = discovery_fraction_by_celltype(
            adata,
            marg_sig_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=counts,
        )
        rows_intersect[trait] = discovery_fraction_by_celltype(
            adata,
            marg_x_cond_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=counts,
        )

        all_signal_ids = valid_signal_values(df_cond[indep_sig_col])
        cond_signal_ids = valid_signal_values(df_cond_sig[indep_sig_col]) if len(df_cond_sig) else []
        n_signals_with_causal = 0

        for signal_id in all_signal_ids:
            signal_mask_all = pd.to_numeric(df_cond[indep_sig_col], errors="coerce").astype("Int64") == signal_id
            sub_all = df_cond.loc[signal_mask_all]
            all_signal_cells = adata.obs_names.intersection(cells_from_metacell_rows(sub_all)) if len(sub_all) else pd.Index([], dtype=str)
            marg_x_all_signal_cells = marg_sig_cells.intersection(all_signal_cells)

            if len(df_cond_sig):
                signal_mask_sig = pd.to_numeric(df_cond_sig[indep_sig_col], errors="coerce").astype("Int64") == signal_id
                sub_sig = df_cond_sig.loc[signal_mask_sig]
            else:
                sub_sig = df_cond_sig

            cond_signal_cells = adata.obs_names.intersection(cells_from_metacell_rows(sub_sig)) if len(sub_sig) else pd.Index([], dtype=str)
            marg_x_cond_signal_cells = marg_sig_cells.intersection(cond_signal_cells)
            if len(marg_x_cond_signal_cells) > 0:
                n_signals_with_causal += 1

            signal_details_rows[(trait, int(signal_id))] = {
                "n_metacells_in_signal_all": int(len(sub_all)),
                "metacell_ids_in_signal_all": join_index(sub_all.index.astype(str)),
                "n_cells_in_signal_all": int(len(all_signal_cells)),
                "cell_ids_in_signal_all": join_index(all_signal_cells),
                "n_marg_x_signal_cells_all": int(len(marg_x_all_signal_cells)),
                "marg_x_signal_cell_ids_all": join_index(marg_x_all_signal_cells),
                "n_metacells_in_signal_cond_sig": int(len(sub_sig)),
                "metacell_ids_in_signal_cond_sig": join_index(sub_sig.index.astype(str)),
                "n_cells_in_signal_cond_sig": int(len(cond_signal_cells)),
                "cell_ids_in_signal_cond_sig": join_index(cond_signal_cells),
                "n_marg_x_signal_cells_cond_sig": int(len(marg_x_cond_signal_cells)),
                "marg_x_signal_cell_ids_cond_sig": join_index(marg_x_cond_signal_cells),
            }

        signal_count_rows[trait] = {
            "n_independent_signals_total": int(len(all_signal_ids)),
            "n_independent_signals_cond_sig": int(len(cond_signal_ids)),
            "n_independent_signals_with_any_causal_cells": int(n_signals_with_causal),
            "n_marginal_sig_cells": int(len(marg_sig_cells)),
            "n_cond_sig_cells": int(len(cond_sig_cells)),
            "n_causal_cells_marg_x_cond": int(len(marg_x_cond_cells)),
            "n_indep_cell_assignments_before_heatmap_filter": int(len(independent_cell_assignments_unfiltered)),
            "n_indep_cell_assignments_saved_after_heatmap_filter": int(len(independent_cell_assignments)),
        }

        if print_summaries:
            print(
                f"[{trait}] signals total={len(all_signal_ids)}, cond-sig={len(cond_signal_ids)}, "
                f"with causal cells={n_signals_with_causal}, causal cells={len(marg_x_cond_cells)}, "
                f"indep assignments raw={len(independent_cell_assignments_unfiltered)}, "
                f"saved in sections >{heatmap_threshold:.1%}={len(independent_cell_assignments)}"
            )

    df_marginal_props = pd.DataFrame.from_dict(rows_marginal, orient="index").astype(float)
    df_marginal_props.index.name = "trait"
    df_marginal_props.columns.name = biocol

    df_marginal_x_cond_props = pd.DataFrame.from_dict(rows_intersect, orient="index").astype(float)
    df_marginal_x_cond_props.index.name = "trait"
    df_marginal_x_cond_props.columns.name = biocol

    df_indep_signal_counts = pd.DataFrame.from_dict(signal_count_rows, orient="index")
    df_indep_signal_counts.index.name = "trait"

    if signal_details_rows:
        df_signal_details = pd.DataFrame.from_dict(signal_details_rows, orient="index")
        df_signal_details.index = pd.MultiIndex.from_tuples(
            df_signal_details.index,
            names=["trait", "independent_signal"],
        )
    else:
        df_signal_details = pd.DataFrame()
        df_signal_details.index = pd.MultiIndex.from_tuples([], names=["trait", "independent_signal"])

    return {
        "df_marginal_props": df_marginal_props,
        "df_marginal_x_cond_props": df_marginal_x_cond_props,
        "df_indep_signal_counts": df_indep_signal_counts,
        "df_signal_details": df_signal_details,
    }

## Heatmap helpers

In [6]:
def aggregate_signal_cells(
    sub_trait: pd.DataFrame,
    raw_signal_id: object,
    *,
    signal_cell_ids_col: str,
) -> pd.Index:
    signal_ids = sub_trait.index.get_level_values(1).astype(str)
    rows = sub_trait.loc[signal_ids == str(raw_signal_id)]
    cells = pd.Index([], dtype=str)
    for value in rows[signal_cell_ids_col].astype(str).tolist():
        cells = cells.append(parse_csv_index(value))
    return pd.Index(cells.astype(str)).unique()


def build_independent_signal_annotations(
    *,
    adata: sc.AnnData,
    df_signal_details: pd.DataFrame | None,
    cell_types: list[str],
    trait_index: list[str],
    biocol: str,
    threshold: float = 0.05,
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
) -> tuple[dict[str, dict[str, str]], set[str]]:
    """
    Build top-right independent-signal annotations for the scDRS-FM heatmap.

    Signals are retained if their marginal-intersect-conditional cells exceed
    `threshold` in at least one plotted cell type. Retained signals are renumbered
    1..K within each trait. For a conditional-associated tile with no exact signal
    hit, the tile is assigned to the retained signal with the most cells of that type.
    """
    ann_map = {trait: {ct: "" for ct in cell_types} for trait in trait_index}
    no_discovery_traits: set[str] = set()

    if df_signal_details is None or len(df_signal_details) == 0:
        no_discovery_traits.update(trait_index)
        return ann_map, no_discovery_traits

    if not isinstance(df_signal_details.index, pd.MultiIndex) or df_signal_details.index.nlevels != 2:
        raise ValueError("df_signal_details must have MultiIndex levels ['trait', 'independent_signal']")
    if signal_cell_ids_col not in df_signal_details.columns:
        raise ValueError(f"df_signal_details missing {signal_cell_ids_col!r}")
    if biocol not in adata.obs.columns:
        raise ValueError(f"adata.obs missing {biocol!r}")

    totals_by_type = adata.obs[biocol].astype(str).value_counts()
    available_traits = set(df_signal_details.index.get_level_values(0).astype(str))

    for trait in trait_index:
        if trait not in available_traits:
            no_discovery_traits.add(trait)
            continue

        sub_trait = df_signal_details.xs(trait, level=0, drop_level=False)
        if "n_marg_x_signal_cells_cond_sig" in sub_trait.columns:
            keep_any = pd.to_numeric(
                sub_trait["n_marg_x_signal_cells_cond_sig"],
                errors="coerce",
            ).fillna(0) > 0
        else:
            keep_any = sub_trait[signal_cell_ids_col].map(lambda x: len(parse_csv_index(x)) > 0)

        sub_any = sub_trait.loc[keep_any]
        if len(sub_any) == 0:
            no_discovery_traits.add(trait)
            continue

        raw_signal_ids = sub_any.index.get_level_values(1).astype(str)
        raw_signal_nums = pd.to_numeric(raw_signal_ids, errors="coerce")
        if raw_signal_nums.notna().all():
            sorted_raw_ids = [str(x) for x in sorted(raw_signal_nums.astype(int).unique().tolist())]
        else:
            sorted_raw_ids = sorted(raw_signal_ids.unique().tolist())

        raw_signal_to_hits: dict[str, set[str]] = {}
        raw_signal_to_counts: dict[str, dict[str, int]] = {}
        eligible_raw_ids: list[str] = []

        for raw_id in sorted_raw_ids:
            cells = aggregate_signal_cells(sub_any, raw_id, signal_cell_ids_col=signal_cell_ids_col)
            cells = adata.obs_names.intersection(cells.astype(str))
            if len(cells) == 0:
                continue

            counts = adata.obs.loc[cells, biocol].astype(str).value_counts().astype(int)
            raw_signal_to_counts[raw_id] = counts.to_dict()

            hit_cell_types = set()
            for ct in cell_types:
                denom = float(totals_by_type.get(ct, 0))
                if denom <= 0:
                    continue
                frac = float(counts.get(ct, 0)) / denom
                if _passes_heatmap_inclusion(frac, threshold):
                    hit_cell_types.add(ct)

            if hit_cell_types:
                raw_signal_to_hits[raw_id] = hit_cell_types
                eligible_raw_ids.append(raw_id)

        if not eligible_raw_ids:
            no_discovery_traits.add(trait)
            continue

        signal_id_map = {raw_id: idx + 1 for idx, raw_id in enumerate(eligible_raw_ids)}

        for ct in cell_types:
            hits = [str(signal_id_map[raw_id]) for raw_id in eligible_raw_ids if ct in raw_signal_to_hits.get(raw_id, set())]
            ann_map[trait][ct] = ",".join(hits)

        # Fallback assignment: for plotted conditional associations that lack a direct hit,
        # pick the eligible signal containing the most cells of that cell type.
        for ct in cell_types:
            if ann_map[trait][ct]:
                continue
            best_raw_id = None
            best_count = -1
            for raw_id in eligible_raw_ids:
                count = int(raw_signal_to_counts.get(raw_id, {}).get(ct, 0))
                if count > best_count:
                    best_count = count
                    best_raw_id = raw_id
            if best_raw_id is not None and best_count > 0:
                ann_map[trait][ct] = str(signal_id_map[best_raw_id])

    return ann_map, no_discovery_traits


def select_marginal_celltypes(
    df_marginal_props: pd.DataFrame,
    *,
    trait_index: list[str],
    threshold: float,
) -> list[str]:
    """Cell types with marginal proportion > threshold for at least one plotted trait."""
    available_traits = [trait for trait in trait_index if trait in df_marginal_props.index]
    if not available_traits:
        return []
    sub = df_marginal_props.loc[available_traits].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return sub.columns[(sub > threshold).any(axis=0)].astype(str).tolist()




def _displayed_scdrsfm_annotation(
    *,
    trait: str,
    cell_type: str,
    conditional_pass: bool,
    ann_map: dict[str, dict[str, str]],
    no_discovery_traits: set[str],
) -> str:
    if not conditional_pass:
        return ""
    annotation = ann_map.get(str(trait), {}).get(str(cell_type), "")
    if annotation:
        return annotation
    if str(trait) in no_discovery_traits:
        return "1"
    return ""


def _build_scdrsfm_heatmap_proportions_table(
    *,
    df_marginal: pd.DataFrame,
    df_conditional: pd.DataFrame,
    adata: sc.AnnData,
    biocol: str,
    trait_labels: dict[str, str],
    marginal_threshold: float,
    conditional_threshold: float,
    ann_map: dict[str, dict[str, str]],
    no_discovery_traits: set[str],
) -> pd.DataFrame:
    if not df_marginal.index.equals(df_conditional.index):
        raise ValueError("Marginal and conditional matrices must have identical trait order.")
    if not df_marginal.columns.equals(df_conditional.columns):
        raise ValueError("Marginal and conditional matrices must have identical cell-type order.")
    traits = df_conditional.index.astype(str).tolist()
    cell_types = df_conditional.columns.astype(str).tolist()
    n_traits = len(traits)
    n_cell_types = len(cell_types)
    repeated_traits = np.repeat(np.asarray(traits, dtype=object), n_cell_types)
    tiled_cell_types = np.tile(np.asarray(cell_types, dtype=object), n_traits)
    marginal_values = df_marginal.astype(float).to_numpy().reshape(-1)
    conditional_values = df_conditional.astype(float).to_numpy().reshape(-1)
    marginal_pass = np.asarray([_passes_heatmap_inclusion(v, marginal_threshold) for v in marginal_values], dtype=bool)
    conditional_pass = np.asarray([_passes_heatmap_inclusion(v, conditional_threshold) for v in conditional_values], dtype=bool)
    counts = celltype_counts(adata, biocol)
    annotations = [ann_map.get(str(trait), {}).get(str(cell_type), "") for trait, cell_type in zip(repeated_traits, tiled_cell_types)]
    displayed_annotations = [
        _displayed_scdrsfm_annotation(
            trait=str(trait),
            cell_type=str(cell_type),
            conditional_pass=bool(passes),
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
        for trait, cell_type, passes in zip(repeated_traits, tiled_cell_types, conditional_pass)
    ]
    return pd.DataFrame({
        "trait_order": np.repeat(np.arange(1, n_traits + 1), n_cell_types),
        "cell_type_order": np.tile(np.arange(1, n_cell_types + 1), n_traits),
        "trait": repeated_traits,
        "trait_label": [trait_labels.get(str(trait), str(trait)) for trait in repeated_traits],
        "cell_type": tiled_cell_types,
        "cell_type_label": tiled_cell_types,
        "cell_type_total_cells": [int(counts.get(str(cell_type), 0)) for cell_type in tiled_cell_types],
        "marginal_cell_type_proportion": marginal_values,
        "marginal_passes_heatmap_inclusion": marginal_pass,
        "conditional_cell_type_proportion": conditional_values,
        "conditional_proportion_displayed": np.where(conditional_pass, conditional_values, 0.0),
        "conditional_passes_heatmap_inclusion": conditional_pass,
        "independent_signals_passing_heatmap_inclusion": annotations,
        "independent_signal_annotation_displayed": displayed_annotations,
        "marginal_threshold": float(marginal_threshold),
        "conditional_threshold": float(conditional_threshold),
        "marginal_inclusion_rule": f"> {float(marginal_threshold):g}",
        "conditional_inclusion_rule": f"> {float(conditional_threshold):g}",
    })


def _write_scdrsfm_heatmap_proportions_csv(*, out_csv: Path | str, **kwargs) -> pd.DataFrame:
    table = _build_scdrsfm_heatmap_proportions_table(**kwargs)
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_csv, index=False)
    print(f"Saved heatmap cell-type proportions: {out_csv} ({len(table):,} rows)")
    return table


def plot_scdrsfm_heatmap(
    *,
    adata: sc.AnnData,
    biocol: str,
    df_marginal_props: pd.DataFrame,
    df_marginal_x_cond_props: pd.DataFrame,
    df_signal_details: pd.DataFrame | None,
    trait_order: list[str] = HEATMAP_TRAITS,
    trait_labels: dict[str, str] = TRAIT_LABELS,
    marginal_threshold: float = 0.05,
    conditional_threshold: float = 0.05,
    signal_cell_ids_col: str = "marg_x_signal_cell_ids_cond_sig",
    title: str = "",
    out_png: Path | str = "scdrsfm_heatmap.png",
    out_csv: Path | str | None = None,
    fontsize_mult: float = 1.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Plot scDRS-FM heatmap using all cell types marginally associated for any plotted trait."""
    trait_index = [trait for trait in trait_order if trait in df_marginal_props.index and trait in df_marginal_x_cond_props.index]
    if not trait_index:
        raise ValueError("No requested traits were present in the scDRS-FM tables.")

    cell_types = select_marginal_celltypes(
        df_marginal_props,
        trait_index=trait_index,
        threshold=marginal_threshold,
    )
    if not cell_types:
        print("No cell types passed the marginal threshold; plotting all shared columns.")
        cell_types = [ct for ct in df_marginal_x_cond_props.columns if ct in df_marginal_props.columns]

    df_marg = df_marginal_props.loc[trait_index, cell_types].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df_cond = df_marginal_x_cond_props.loc[trait_index, cell_types].apply(pd.to_numeric, errors="coerce").fillna(0.0)

    n_traits, n_celltypes = df_cond.shape
    counts = celltype_counts(adata, biocol)

    ann_map, no_discovery_traits = build_independent_signal_annotations(
        adata=adata,
        df_signal_details=df_signal_details,
        cell_types=cell_types,
        trait_index=trait_index,
        biocol=biocol,
        threshold=conditional_threshold,
        signal_cell_ids_col=signal_cell_ids_col,
    )

    fig, ax, ax_box = create_square_heatmap_figure(n_traits, n_celltypes)
    boundaries = np.linspace(0, 1, 12)
    cmap = ListedColormap(sns.color_palette("Blues", n_colors=11))
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    star_size = 46 * fontsize_mult
    ann_size = 15 * fontsize_mult

    for trait_idx, trait in enumerate(df_cond.index.astype(str)):
        for ct_idx, ct in enumerate(cell_types):
            raw_marginal = float(df_marg.loc[trait, ct])
            raw_conditional = float(df_cond.loc[trait, ct])
            display_value = raw_conditional if _passes_heatmap_inclusion(raw_conditional, conditional_threshold) else 0.0

            x = ct_idx
            y = n_traits - trait_idx - 1
            facecolor = "white" if display_value == 0.0 else cmap(norm(display_value))
            ax.add_patch(Rectangle((x, y), 1, 1, facecolor=facecolor, edgecolor="none"))

            has_marginal = _passes_heatmap_inclusion(raw_marginal, marginal_threshold)
            has_conditional = _passes_heatmap_inclusion(raw_conditional, conditional_threshold)

            if has_marginal:
                ax.text(x + 0.5, y + 0.5, "☆", ha="center", va="center", fontsize=star_size,
                        color="black", fontweight="bold", zorder=20)
            if has_conditional:
                ax.text(x + 0.5, y + 0.5, "★", ha="center", va="center", fontsize=star_size * 0.82,
                        color="red", fontweight="bold",
                        path_effects=[pe.withStroke(linewidth=2, foreground="black")], zorder=21)

            annotation = ann_map.get(trait, {}).get(ct, "")
            if has_conditional and annotation:
                ax.text(x + 0.95, y + 0.95, annotation, ha="right", va="top",
                        fontsize=ann_size, color="white", fontweight="bold",
                        path_effects=[pe.withStroke(linewidth=2.8, foreground="black")], zorder=30)
            elif has_conditional and trait in no_discovery_traits:
                ax.text(x + 0.95, y + 0.95, "1", ha="right", va="top",
                        fontsize=ann_size, color="white", fontweight="bold",
                        path_effects=[pe.withStroke(linewidth=2.8, foreground="black")], zorder=30)

    ax.set_xlim(0, n_celltypes)
    ax.set_ylim(0, n_traits)
    ax.set_xticks(np.arange(n_celltypes) + 0.5)
    ax.set_yticks(np.arange(n_traits) + 0.5)
    ax.set_xticks(np.arange(n_celltypes + 1), minor=True)
    ax.set_yticks(np.arange(n_traits + 1), minor=True)
    ax.grid(which="minor", color="lightgray", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_xticklabels(label_celltypes_with_counts(cell_types, counts), fontsize=16 * fontsize_mult, rotation=45, ha="right")
    traits_bottom_to_top = list(reversed(trait_index))
    y_labels = [trait_labels.get(trait, trait) for trait in traits_bottom_to_top]
    ax.set_yticklabels(y_labels, fontsize=18 * fontsize_mult)
    color_ticklabels(ax.get_yticklabels(), [trait_color(t) for t in traits_bottom_to_top], fontsize=18 * fontsize_mult)

    color_counts, segment_names = trait_group_segments_bottom_to_top(trait_index)
    add_right_group_lines(ax, color_counts, segment_names, fontsize=24 * fontsize_mult)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    ax_left, ax_bottom, ax_width, ax_height = ax_box
    cbar_ax = fig.add_axes([ax_left, min(0.95, ax_bottom + ax_height + 0.19), min(0.32, ax_width * 0.65), 0.018])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", boundaries=boundaries,
                        ticks=[0, 0.5, 1.0], spacing="proportional", drawedges=True)
    cbar.ax.set_xticklabels(["0%", "50%", "100%"], fontsize=14 * fontsize_mult)
    cbar.set_label("Prop. sig. conditional cells", fontsize=16 * fontsize_mult, labelpad=12 * fontsize_mult)

    inferred_handle = Line2D([], [], linestyle="None", marker="$1$", color="white", markersize=15 * fontsize_mult)
    inferred_handle.set_path_effects([pe.withStroke(linewidth=2.8, foreground="black")])
    legend_elements = [
        Line2D([], [], marker="*", linestyle="None", markerfacecolor="none", markeredgecolor="black",
               markeredgewidth=2, markersize=20 * fontsize_mult),
        Line2D([], [], marker="*", linestyle="None", markerfacecolor="red", markeredgecolor="black",
               markersize=20 * fontsize_mult),
        inferred_handle,
    ]
    fig.legend(
        legend_elements,
        ["Marginal association", "Conditional association", "Inferred cell population"],
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        prop={"size": 16 * fontsize_mult},
        frameon=True,
    )

    if title:
        fig.suptitle(title, fontsize=22 * fontsize_mult, y=0.985)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    if out_csv is not None:
        _write_scdrsfm_heatmap_proportions_csv(
            out_csv=out_csv,
            df_marginal=df_marg,
            df_conditional=df_cond,
            adata=adata,
            biocol=biocol,
            trait_labels=trait_labels,
            marginal_threshold=marginal_threshold,
            conditional_threshold=conditional_threshold,
            ann_map=ann_map,
            no_discovery_traits=no_discovery_traits,
        )
    plt.show()
    print(f"Saved {out_png}")
    return df_marg, df_cond

## scDRS and scPagwas parsing / heatmaps

In [7]:
def build_marginal_props_from_score_files(
    *,
    adata: sc.AnnData,
    results_dir: Path,
    traits: list[str],
    biocol: str,
    fdr_alpha: float = 0.1,
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    file_suffix: str = ".marginal_score.gz",
) -> pd.DataFrame:
    """Fraction of cell-level significant marginal cells per cell type."""
    counts = celltype_counts(adata, biocol)
    celltype_order = counts.index.tolist()
    rows = {}

    for trait in traits:
        prefix = Path(trait).name
        score_file = Path(results_dir) / f"{prefix}{file_suffix}"
        if not score_file.exists():
            raise FileNotFoundError(f"Missing score file: {score_file}")
        df = pd.read_csv(score_file, sep="\t", compression="infer", index_col=0)
        pcol = pick_first_existing_col(df, pval_col_candidates, what="marginal score p-value")
        sig_mask = bh_fdr_mask(df[pcol].to_numpy(), alpha=fdr_alpha)
        sig_cells = adata.obs_names.intersection(df.index[sig_mask].astype(str))
        rows[trait] = discovery_fraction_by_celltype(
            adata,
            sig_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=counts,
        )

    out = pd.DataFrame.from_dict(rows, orient="index").astype(float)
    out.index.name = "trait"
    out.columns.name = biocol
    return out


def build_scdrs_celltype_association_matrix(
    *,
    results_dir: Path,
    traits: list[str],
    biocol: str = "Cell_population",
    alpha: float = 0.05,
    pval_col: str = "assoc_mcp",
    file_suffix_template: str = ".marginal_score.gz.scdrs_ct.{biocol}",
) -> pd.DataFrame:
    """Cell type association matrix from scDRS cell-type files using BH FDR on assoc_mcp."""
    rows: dict[str, pd.Series] = {}
    all_celltypes: list[str] = []

    for trait in traits:
        prefix = Path(trait).name
        suffix = file_suffix_template.format(biocol=biocol)
        path = Path(results_dir) / f"{prefix}{suffix}"
        if not path.exists():
            raise FileNotFoundError(f"Missing scDRS cell-type association file: {path}")
        df = pd.read_csv(path, sep="\t", index_col=0)
        if pval_col not in df.columns:
            raise ValueError(f"{path} missing {pval_col!r}; columns={df.columns.tolist()}")
        sig = bh_fdr_mask(df[pval_col].to_numpy(), alpha=alpha)
        s = pd.Series(sig, index=df.index.astype(str), name=trait)
        rows[trait] = s
        all_celltypes.extend(s.index.tolist())

    ordered_celltypes = pd.Index(all_celltypes).drop_duplicates().tolist()
    assoc = pd.DataFrame.from_dict(rows, orient="index").reindex(columns=ordered_celltypes, fill_value=False)
    assoc = assoc.astype(bool)
    assoc.index.name = "trait"
    assoc.columns.name = biocol
    return assoc


def scpagwas_singlecell_file(base_dir: Path, trait: str) -> Path:
    prefix = Path(trait).name
    candidates = [
        Path(base_dir) / prefix / f"{prefix}_snglecell_scPagwas_score_pvalue.Result.csv",
        Path(base_dir) / prefix / f"{prefix}_singlecell_scPagwas_score_pvalue.Result.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def scpagwas_celltype_file(base_dir: Path, trait: str) -> Path:
    prefix = Path(trait).name
    return Path(base_dir) / prefix / f"{prefix}_Merged_celltype_pvalue.csv"


def build_scpagwas_tables(
    *,
    adata: sc.AnnData,
    base_dir: Path,
    traits: list[str],
    biocol: str = "Cell_population",
    cell_alpha: float = 0.1,
    ct_alpha: float = 0.05,
    score_sig_col: str = "Random_Correct_BG_adjp",
    ct_pval_col: str = "pvalue",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build scPagwas cell-level significant-cell fractions and cell-type associations.

    Cells: Random_Correct_BG_adjp < cell_alpha.
    Cell types: BH FDR on merged cell-type p-values at ct_alpha.
    """
    counts = celltype_counts(adata, biocol)
    celltype_order = counts.index.tolist()
    prop_rows: dict[str, pd.Series] = {}
    assoc_rows: dict[str, pd.Series] = {}
    assoc_celltypes: list[str] = []

    for trait in traits:
        score_file = scpagwas_singlecell_file(base_dir, trait)
        ct_file = scpagwas_celltype_file(base_dir, trait)
        if not score_file.exists():
            raise FileNotFoundError(f"Missing scPagwas single-cell file: {score_file}")
        if not ct_file.exists():
            raise FileNotFoundError(f"Missing scPagwas merged cell-type file: {ct_file}")

        df_score = pd.read_csv(score_file, index_col=0)
        if score_sig_col not in df_score.columns:
            raise ValueError(f"{score_file} missing {score_sig_col!r}; columns={df_score.columns.tolist()}")
        sig_cells = adata.obs_names.intersection(df_score.index[pd.to_numeric(df_score[score_sig_col], errors="coerce") < cell_alpha].astype(str))
        prop_rows[trait] = discovery_fraction_by_celltype(
            adata,
            sig_cells,
            biocol=biocol,
            celltype_order=celltype_order,
            totals_by_type=counts,
        )

        df_ct = pd.read_csv(ct_file, index_col=0)
        if "celltype" not in df_ct.columns or ct_pval_col not in df_ct.columns:
            raise ValueError(f"{ct_file} must contain 'celltype' and {ct_pval_col!r}")
        sig_ct_mask = bh_fdr_mask(df_ct[ct_pval_col].to_numpy(), alpha=ct_alpha)
        s = pd.Series(sig_ct_mask, index=df_ct["celltype"].astype(str), name=trait)
        assoc_rows[trait] = s
        assoc_celltypes.extend(s.index.tolist())

    df_props = pd.DataFrame.from_dict(prop_rows, orient="index").astype(float)
    df_props.index.name = "trait"
    df_props.columns.name = biocol

    assoc_order = pd.Index(assoc_celltypes).drop_duplicates().tolist()
    df_assoc = pd.DataFrame.from_dict(assoc_rows, orient="index").reindex(columns=assoc_order, fill_value=False).astype(bool)
    df_assoc.index.name = "trait"
    df_assoc.columns.name = biocol
    return df_props, df_assoc




def _build_celltype_association_heatmap_proportions_table(
    *,
    df_prop: pd.DataFrame,
    df_assoc: pd.DataFrame,
    adata: sc.AnnData,
    biocol: str,
    trait_labels: dict[str, str],
) -> pd.DataFrame:
    if not df_prop.index.equals(df_assoc.index) or not df_prop.columns.equals(df_assoc.columns):
        raise ValueError("Proportion and association matrices must have identical plotted order.")
    traits = df_prop.index.astype(str).tolist()
    cell_types = df_prop.columns.astype(str).tolist()
    n_traits = len(traits)
    n_cell_types = len(cell_types)
    repeated_traits = np.repeat(np.asarray(traits, dtype=object), n_cell_types)
    tiled_cell_types = np.tile(np.asarray(cell_types, dtype=object), n_traits)
    values = df_prop.astype(float).to_numpy().reshape(-1)
    associated = df_assoc.astype(bool).to_numpy().reshape(-1)
    counts = celltype_counts(adata, biocol)
    return pd.DataFrame({
        "trait_order": np.repeat(np.arange(1, n_traits + 1), n_cell_types),
        "cell_type_order": np.tile(np.arange(1, n_cell_types + 1), n_traits),
        "trait": repeated_traits,
        "trait_label": [trait_labels.get(str(trait), str(trait)) for trait in repeated_traits],
        "cell_type": tiled_cell_types,
        "cell_type_label": tiled_cell_types,
        "cell_type_total_cells": [int(counts.get(str(cell_type), 0)) for cell_type in tiled_cell_types],
        "cell_type_proportion": values,
        "cell_type_proportion_displayed": values,
        "association_star_displayed": associated,
    })


def _write_celltype_association_heatmap_proportions_csv(*, out_csv: Path | str, **kwargs) -> pd.DataFrame:
    table = _build_celltype_association_heatmap_proportions_table(**kwargs)
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_csv, index=False)
    print(f"Saved heatmap cell-type proportions: {out_csv} ({len(table):,} rows)")
    return table


def plot_celltype_association_heatmap(
    *,
    adata: sc.AnnData,
    biocol: str,
    prop_df: pd.DataFrame,
    assoc_df: pd.DataFrame,
    trait_order: list[str] = HEATMAP_TRAITS,
    trait_labels: dict[str, str] = TRAIT_LABELS,
    title: str = "",
    colorbar_label: str = "Prop. sig. marginal cells",
    star_label: str = "Cell-type association",
    out_png: Path | str = "celltype_association_heatmap.png",
    out_csv: Path | str | None = None,
    fontsize_mult: float = 1.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Generic marginal-only heatmap with red stars for associated cell types."""
    trait_index = [trait for trait in trait_order if trait in prop_df.index and trait in assoc_df.index]
    if not trait_index:
        raise ValueError("No requested traits were present in both prop_df and assoc_df.")

    associated_celltypes = assoc_df.loc[trait_index].columns[assoc_df.loc[trait_index].any(axis=0)].astype(str).tolist()
    associated_celltypes = [ct for ct in associated_celltypes if ct in prop_df.columns]
    if not associated_celltypes:
        raise ValueError("No cell type associations found for any requested trait.")

    df_prop = prop_df.loc[trait_index, associated_celltypes].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df_assoc = assoc_df.loc[trait_index, associated_celltypes].astype(bool)

    n_traits, n_celltypes = df_prop.shape
    counts = celltype_counts(adata, biocol)

    fig, ax, ax_box = create_square_heatmap_figure(n_traits, n_celltypes)
    boundaries = np.linspace(0, 1, 12)
    cmap = ListedColormap(sns.color_palette("Blues", n_colors=11))
    norm = BoundaryNorm(boundaries, ncolors=cmap.N, clip=True)

    for trait_idx, trait in enumerate(df_prop.index.astype(str)):
        for ct_idx, ct in enumerate(associated_celltypes):
            value = float(df_prop.loc[trait, ct])
            x = ct_idx
            y = n_traits - trait_idx - 1
            facecolor = "white" if value <= 0 else cmap(norm(value))
            ax.add_patch(Rectangle((x, y), 1, 1, facecolor=facecolor, edgecolor="none"))
            if bool(df_assoc.loc[trait, ct]):
                ax.text(x + 0.5, y + 0.5, "★", ha="center", va="center",
                        fontsize=38 * fontsize_mult, color="red", fontweight="bold",
                        path_effects=[pe.withStroke(linewidth=2, foreground="black")], zorder=20)

    ax.set_xlim(0, n_celltypes)
    ax.set_ylim(0, n_traits)
    ax.set_xticks(np.arange(n_celltypes) + 0.5)
    ax.set_yticks(np.arange(n_traits) + 0.5)
    ax.set_xticks(np.arange(n_celltypes + 1), minor=True)
    ax.set_yticks(np.arange(n_traits + 1), minor=True)
    ax.grid(which="minor", color="lightgray", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_xticklabels(label_celltypes_with_counts(associated_celltypes, counts), fontsize=16 * fontsize_mult, rotation=45, ha="right")
    traits_bottom_to_top = list(reversed(trait_index))
    ax.set_yticklabels([trait_labels.get(t, t) for t in traits_bottom_to_top], fontsize=18 * fontsize_mult)
    color_ticklabels(ax.get_yticklabels(), [trait_color(t) for t in traits_bottom_to_top], fontsize=18 * fontsize_mult)

    color_counts, segment_names = trait_group_segments_bottom_to_top(trait_index)
    add_right_group_lines(ax, color_counts, segment_names, fontsize=24 * fontsize_mult)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    ax_left, ax_bottom, ax_width, ax_height = ax_box
    cbar_ax = fig.add_axes([ax_left, min(0.95, ax_bottom + ax_height + 0.19), min(0.32, ax_width * 0.65), 0.018])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", boundaries=boundaries,
                        ticks=[0, 0.5, 1.0], spacing="proportional", drawedges=True)
    cbar.ax.set_xticklabels(["0%", "50%", "100%"], fontsize=14 * fontsize_mult)
    cbar.set_label(colorbar_label, fontsize=16 * fontsize_mult, labelpad=12 * fontsize_mult)

    legend_elements = [
        Line2D([], [], marker="*", linestyle="None", markerfacecolor="red", markeredgecolor="black",
               markersize=20 * fontsize_mult, label=star_label),
    ]
    fig.legend(legend_elements, [star_label], loc="upper right", bbox_to_anchor=(0.98, 0.98),
               prop={"size": 16 * fontsize_mult}, frameon=True)

    if title:
        fig.suptitle(title, fontsize=22 * fontsize_mult, y=0.985)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    if out_csv is not None:
        _write_celltype_association_heatmap_proportions_csv(
            out_csv=out_csv,
            df_prop=df_prop,
            df_assoc=df_assoc,
            adata=adata,
            biocol=biocol,
            trait_labels=trait_labels,
        )
    plt.show()
    print(f"Saved {out_png}")
    return df_prop, df_assoc

## UMAP helpers

In [8]:

def prepare_umap(
    adata: sc.AnnData,
    *,
    n_neighbors: int = 15,
    n_pcs: int = 40,
    max_pcs: int = 50,
    force: bool = False,
) -> sc.AnnData:
    """Return a copy with X_umap present."""
    adata_umap = adata.copy()
    if ("X_umap" in adata_umap.obsm) and not force:
        return adata_umap

    sc.pp.highly_variable_genes(
        adata_umap,
        subset=False,
        min_disp=0.5,
        min_mean=0.0125,
        max_mean=10,
        n_bins=20,
        n_top_genes=None,
    )
    sc.pp.scale(adata_umap, max_value=10, zero_center=False)
    sc.pp.pca(
        adata_umap,
        n_comps=min(adata_umap.n_obs, max_pcs),
        use_highly_variable=True,
        svd_solver="arpack",
    )
    sc.pp.neighbors(
        adata_umap,
        n_neighbors=min(adata_umap.n_obs, n_neighbors),
        n_pcs=min(adata_umap.n_obs, n_pcs),
    )
    sc.tl.umap(adata_umap)
    return adata_umap


def clean_umap_axis(ax):
    """Remove axes, ticks, and spines from a UMAP axis."""
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def read_tsv_score_series(score_file: Path, cells: pd.Index, *, score_col: str = "norm_score") -> pd.Series:
    df = pd.read_csv(score_file, sep="\t", compression="infer", index_col=0)
    if score_col not in df.columns:
        raise ValueError(f"{score_file} missing {score_col!r}; columns={df.columns.tolist()}")
    return pd.to_numeric(df.reindex(cells)[score_col], errors="coerce")


def read_csv_score_series(score_file: Path, cells: pd.Index, *, score_col: str) -> pd.Series:
    df = pd.read_csv(score_file, index_col=0)
    if score_col not in df.columns:
        raise ValueError(f"{score_file} missing {score_col!r}; columns={df.columns.tolist()}")
    return pd.to_numeric(df.reindex(cells)[score_col], errors="coerce")


def read_scdrs_like_score_and_sig(
    score_file: Path,
    cells: pd.Index,
    *,
    score_col: str = "norm_score",
    pval_col_candidates: Sequence[str] = ("pval", "mc_pval"),
    fdr_alpha: float = 0.1,
) -> tuple[pd.Series, pd.Series]:
    """Read a scDRS/scDRS-FM score file and return aligned scores plus BH-FDR significant-cell mask."""
    df = pd.read_csv(score_file, sep="\t", compression="infer", index_col=0)
    if score_col not in df.columns:
        raise ValueError(f"{score_file} missing {score_col!r}; columns={df.columns.tolist()}")
    pcol = pick_first_existing_col(df, pval_col_candidates, what=f"{score_file} score p-value")
    sig = pd.Series(bh_fdr_mask(df[pcol].to_numpy(), alpha=fdr_alpha), index=df.index.astype(str))
    scores = pd.to_numeric(df[score_col], errors="coerce")
    scores.index = scores.index.astype(str)
    return scores.reindex(cells), sig.reindex(cells, fill_value=False).astype(bool)


def read_scpagwas_score_and_sig(
    score_file: Path,
    cells: pd.Index,
    *,
    score_col: str = "scPagwas.TRS.Score",
    sig_col: str = "Random_Correct_BG_adjp",
    sig_alpha: float = 0.1,
) -> tuple[pd.Series, pd.Series]:
    """Read a scPagwas single-cell score file and return aligned scores plus significant-cell mask."""
    df = pd.read_csv(score_file, index_col=0)
    if score_col not in df.columns or sig_col not in df.columns:
        raise ValueError(f"{score_file} must contain {score_col!r} and {sig_col!r}; columns={df.columns.tolist()}")
    scores = pd.to_numeric(df[score_col], errors="coerce")
    sig = pd.to_numeric(df[sig_col], errors="coerce") < sig_alpha
    scores.index = scores.index.astype(str)
    sig.index = sig.index.astype(str)
    return scores.reindex(cells), sig.reindex(cells, fill_value=False).astype(bool)


def plot_cell_population_umap(
    adata_umap: sc.AnnData,
    *,
    key: str,
    title: str,
    out_png: Path | str,
    point_size: float = 6,
    label_fontsize: float = 14,
):
    """CD4 T-cell UMAP with on-data cell-population labels, cleaned from the original Soskic notebook."""
    try:
        from adjustText import adjust_text
    except Exception:
        adjust_text = None

    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] is missing. Run prepare_umap first.")
    if key not in adata_umap.obs.columns:
        raise KeyError(f"{key!r} not found in adata_umap.obs")

    if not isinstance(adata_umap.obs[key].dtype, pd.CategoricalDtype):
        adata_umap.obs[key] = pd.Categorical(adata_umap.obs[key].astype(str))

    clusters = adata_umap.obs[key].cat.categories
    palette = sns.color_palette("husl", n_colors=len(clusters)).as_hex()
    color_dict = dict(zip(clusters, palette))
    adata_umap.uns[f"{key}_colors"] = [color_dict[c] for c in clusters]

    fig, ax = plt.subplots(figsize=(6.5, 8))
    sc.pl.umap(
        adata_umap,
        color=key,
        ax=ax,
        show=False,
        legend_loc="on data",
        size=point_size,
        title=None,
        frameon=False,
    )

    ax.set_title(title, fontsize=32)
    clean_umap_axis(ax)

    texts = []
    for text in ax.texts:
        label = text.get_text()
        if label in color_dict:
            text.set_text(f"● {label}")
            text.set_color(color_dict[label])
            text.set_fontsize(label_fontsize)
            text.set_bbox(dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.2", linewidth=0.8))
            texts.append(text)

    if adjust_text is not None and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="->", color="black", lw=1.5))

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {out_png}")


def descending_score_order(values: Sequence[float], mask: Sequence[bool] | None = None) -> np.ndarray:
    """Return stable point indices ordered by score from highest to lowest, with NaNs last."""
    values_arr = np.asarray(values, dtype=float)
    indices = np.arange(values_arr.size)
    if mask is not None:
        mask_arr = np.asarray(mask, dtype=bool)
        if mask_arr.shape != values_arr.shape:
            raise ValueError("mask and values must have the same shape")
        indices = indices[mask_arr]
    if indices.size == 0:
        return indices

    sortable = np.where(np.isnan(values_arr[indices]), -np.inf, values_arr[indices])
    return indices[np.argsort(-sortable, kind="mergesort")]


def plot_one_score_umap(
    adata_umap: sc.AnnData,
    scores: pd.Series,
    *,
    title: str,
    out_png: Path | str,
    sig_mask: pd.Series | None = None,
    cmap: str = "RdBu_r",
    center_at: float = 0.0,
    base_size: float = 6,
    sig_size: float = 22,
    base_alpha: float = 0.35,
    sig_alpha: float = 0.85,
    sort_descending: bool = False,
    show: bool = True,
):
    """Score UMAP, optionally ordered by descending score and highlighting significant cells."""
    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] is missing. Run prepare_umap first.")

    scores = pd.to_numeric(scores.reindex(adata_umap.obs_names), errors="coerce")
    values = scores.to_numpy()
    sig = (
        np.zeros(adata_umap.n_obs, dtype=bool)
        if sig_mask is None
        else sig_mask.reindex(adata_umap.obs_names, fill_value=False).astype(bool).to_numpy()
    )

    finite = np.isfinite(values)
    max_abs = float(np.nanmax(np.abs(values[finite] - center_at))) if finite.any() else 1.0
    if max_abs == 0:
        max_abs = 1.0
    norm = mcolors.TwoSlopeNorm(
        vcenter=center_at,
        vmin=center_at - max_abs,
        vmax=center_at + max_abs,
    )

    U = adata_umap.obsm["X_umap"]
    x, y = U[:, 0], U[:, 1]
    background_order = (
        descending_score_order(values, ~sig)
        if sort_descending
        else np.flatnonzero(~sig)
    )
    sig_order = (
        descending_score_order(values, sig)
        if sort_descending
        else np.flatnonzero(sig)
    )

    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    ax.scatter(
        x[background_order], y[background_order],
        c=values[background_order], cmap=cmap, norm=norm,
        s=base_size, alpha=base_alpha if sig_mask is not None else 1.0,
        linewidths=0,
    )
    if sig_order.size:
        ax.scatter(
            x[sig_order], y[sig_order],
            c=values[sig_order], cmap=cmap, norm=norm,
            s=sig_size, alpha=sig_alpha,
            edgecolors="black", linewidths=0.45,
        )

    ax.set_title(title, fontsize=18)
    clean_umap_axis(ax)

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=11)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    print(f"Saved {out_png}")


def plot_score_umap_panels_highlight(
    adata_umap: sc.AnnData,
    score_data_by_title: dict[str, tuple[pd.Series, pd.Series | None]],
    *,
    out_png: Path | str,
    cmap: str = "RdBu_r",
    center_at: float = 0.0,
    base_size: float = 6,
    sig_size: float = 22,
    base_alpha: float = 0.35,
    sig_alpha: float = 0.85,
    sort_descending: bool = False,
    show: bool = True,
):
    """Multi-panel score UMAP with significant cells emphasized in each panel."""
    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] is missing. Run prepare_umap first.")

    titles = list(score_data_by_title.keys())
    n = len(titles)
    U = adata_umap.obsm["X_umap"]
    x, y = U[:, 0], U[:, 1]

    fig, axes = plt.subplots(1, n, figsize=(5.6 * n, 5.2), constrained_layout=True)
    if n == 1:
        axes = [axes]

    for ax, title in zip(axes, titles):
        scores, sig_mask = score_data_by_title[title]
        values = pd.to_numeric(scores.reindex(adata_umap.obs_names), errors="coerce").to_numpy()
        sig = (
            np.zeros(adata_umap.n_obs, dtype=bool)
            if sig_mask is None
            else sig_mask.reindex(adata_umap.obs_names, fill_value=False).astype(bool).to_numpy()
        )

        finite = np.isfinite(values)
        max_abs = float(np.nanmax(np.abs(values[finite] - center_at))) if finite.any() else 1.0
        if max_abs == 0:
            max_abs = 1.0
        norm = mcolors.TwoSlopeNorm(
            vcenter=center_at,
            vmin=center_at - max_abs,
            vmax=center_at + max_abs,
        )

        background_order = (
            descending_score_order(values, ~sig)
            if sort_descending
            else np.flatnonzero(~sig)
        )
        sig_order = (
            descending_score_order(values, sig)
            if sort_descending
            else np.flatnonzero(sig)
        )

        ax.scatter(
            x[background_order], y[background_order],
            c=values[background_order], cmap=cmap, norm=norm,
            s=base_size, alpha=base_alpha, linewidths=0,
        )
        if sig_order.size:
            ax.scatter(
                x[sig_order], y[sig_order],
                c=values[sig_order], cmap=cmap, norm=norm,
                s=sig_size, alpha=sig_alpha, edgecolors="black", linewidths=0.45,
            )

        ax.set_title(title, fontsize=16)
        clean_umap_axis(ax)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=10)

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    print(f"Saved {out_png}")


def explode_cell_ids(df: pd.DataFrame, *, cell_ids_col: str) -> pd.DataFrame:
    """Explode comma-separated cell_ids from scDRS-FM conditional rows."""
    long = df.copy()
    long["cell_id"] = long[cell_ids_col].astype(str).str.split(",")
    long = long.explode("cell_id", ignore_index=True)
    long["cell_id"] = long["cell_id"].astype(str).str.strip()
    long = long[(long["cell_id"] != "") & (long["cell_id"] != "nan")]
    return long


def assign_conditional_scores_to_cells_filtered_signals(
    *,
    adata_umap: sc.AnnData,
    out_folder: Path,
    trait: str,
    indep_sig_col: str = "independent_signal",
    cell_ids_col: str = "cell_ids",
    score_col_candidates: Sequence[str] = ("tagging_score", "conditional_score", "score", "z", "zscore", "stat"),
    pval_col_candidates_cond: Sequence[str] = ("pval", "mc_pval"),
    pval_col_candidates_marg: Sequence[str] = ("pval", "mc_pval"),
    cond_file_suffix: str = ".conditional.tagging_score.gz",
    marg_file_suffix: str = ".marginal_score.gz",
    marginal_metacell_col: str = "metacell",
    fdr_alpha: float = 0.1,
    min_causal_cells_per_signal: int = 100,
    cellpop_key: str = "Cell_population",
    min_fraction_within_any_cellpop: float = 0.05,
    obs_score_key: Optional[str] = None,
    obs_sig_key: Optional[str] = None,
    overwrite: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[int, int]]:
    """
    Assign one conditional disease score per cell and filtered independent population IDs.

    A retained independent population must have enough marginal∩conditional cells and account
    for at least `min_fraction_within_any_cellpop` of one Soskic cell population.
    """
    prefix = Path(trait).name
    out_folder = Path(out_folder)
    cond_file = out_folder / f"{prefix}{cond_file_suffix}"
    marg_file = out_folder / f"{prefix}{marg_file_suffix}"

    df_cond = pd.read_csv(cond_file, sep="\t", compression="infer", index_col=0)
    df_marg = pd.read_csv(marg_file, sep="\t", compression="infer", index_col=0)

    if cellpop_key not in adata_umap.obs.columns:
        raise ValueError(f"adata_umap.obs missing {cellpop_key!r}")
    if cell_ids_col not in df_cond.columns or indep_sig_col not in df_cond.columns:
        raise ValueError(f"{cond_file} must contain {cell_ids_col!r} and {indep_sig_col!r}")
    if marginal_metacell_col not in df_marg.columns:
        raise ValueError(f"{marg_file} missing {marginal_metacell_col!r}")

    score_col = pick_first_existing_col(df_cond, score_col_candidates, what="conditional score")
    pcol_cond = pick_first_existing_col(df_cond, pval_col_candidates_cond, what="conditional p-value")
    pcol_marg = pick_first_existing_col(df_marg, pval_col_candidates_marg, what="marginal p-value")

    obs_names = adata_umap.obs_names.astype(str)

    marg_sig_mask = bh_fdr_mask(df_marg[pcol_marg].to_numpy(), alpha=fdr_alpha)
    marg_sig_cells = obs_names.intersection(df_marg.index[marg_sig_mask].astype(str))

    cond_sig_mask = bh_fdr_mask(df_cond[pcol_cond].to_numpy(), alpha=fdr_alpha)
    df_cond_sig = df_cond.loc[cond_sig_mask] if cond_sig_mask.any() else df_cond.iloc[0:0]

    long_all = explode_cell_ids(df_cond[[score_col, indep_sig_col, cell_ids_col]], cell_ids_col=cell_ids_col)
    long_all = long_all[long_all["cell_id"].isin(obs_names)]

    long_sig = (
        explode_cell_ids(df_cond_sig[[score_col, indep_sig_col, cell_ids_col]], cell_ids_col=cell_ids_col)
        if len(df_cond_sig) else long_all.iloc[0:0].copy()
    )
    long_sig = long_sig[long_sig["cell_id"].isin(obs_names)]

    duplicated = long_all["cell_id"].duplicated(keep=False)
    if duplicated.any():
        examples = long_all.loc[duplicated, ["cell_id", indep_sig_col]].head(10)
        raise ValueError(f"A cell appears in multiple conditional metacells. Examples:\n{examples}")

    long_sig[indep_sig_col] = pd.to_numeric(long_sig[indep_sig_col], errors="coerce")
    long_sig = long_sig[long_sig[indep_sig_col].notna()]
    long_sig[indep_sig_col] = long_sig[indep_sig_col].astype(int)
    long_sig = long_sig[long_sig[indep_sig_col] >= 0]

    causal_cells_by_sig: dict[int, pd.Index] = {}
    causal_counts: dict[int, int] = {}
    
    for sig, sub in long_sig.groupby(indep_sig_col, sort=True):
        cond_cells = pd.Index(sub["cell_id"].astype(str)).unique()
    
        # Keep only cells significant in both the marginal and conditional analyses.
        # This restores the original independent-population definition.
        causal_cells = marg_sig_cells.intersection(cond_cells)
    
        sig = int(sig)
        causal_cells_by_sig[sig] = causal_cells
        causal_counts[sig] = int(len(causal_cells))
    

    cellpop = adata_umap.obs[cellpop_key].astype(str).copy()
    cellpop.index = obs_names
    cellpop_totals = cellpop.value_counts()

    max_frac_by_sig: dict[int, float] = {}
    for sig, causal_cells in causal_cells_by_sig.items():
        if len(causal_cells) == 0:
            max_frac_by_sig[sig] = 0.0
            continue
        counts_in_sig = cellpop.loc[cellpop.index.intersection(causal_cells)].value_counts()
        if counts_in_sig.empty:
            max_frac_by_sig[sig] = 0.0
            continue
        max_frac_by_sig[sig] = float((counts_in_sig / cellpop_totals.loc[counts_in_sig.index]).max())

    kept_old_sigs = sorted(
        s for s, n in causal_counts.items()
        if n >= min_causal_cells_per_signal and max_frac_by_sig.get(s, 0.0) >= min_fraction_within_any_cellpop
    )
    remap = {old: idx + 1 for idx, old in enumerate(kept_old_sigs)}

    score_key = obs_score_key or f"{prefix}_conditional_score"
    sig_key = obs_sig_key or f"{prefix}_{indep_sig_col}_filtered"
    if (not overwrite) and (score_key in adata_umap.obs.columns or sig_key in adata_umap.obs.columns):
        raise ValueError(f"Refusing to overwrite existing columns: {score_key}, {sig_key}")

    score_s = pd.Series(np.nan, index=obs_names, dtype=float)
    score_s.loc[long_all["cell_id"].values] = pd.to_numeric(long_all[score_col].values, errors="coerce")

    sig_s = pd.Series(-1, index=obs_names, dtype=int)
    for old_sig in kept_old_sigs:
        causal_cells = causal_cells_by_sig.get(old_sig, pd.Index([], dtype=str))
        if len(causal_cells):
            sig_s.loc[obs_names.intersection(causal_cells)] = remap[old_sig]

    adata_umap.obs[score_key] = score_s.reindex(adata_umap.obs_names).values
    adata_umap.obs[sig_key] = pd.Categorical(
        sig_s.reindex(adata_umap.obs_names).astype(int),
        categories=[-1] + list(range(1, len(kept_old_sigs) + 1)),
        ordered=True,
    )
    return df_cond, df_marg, remap


def plot_ibd_scores_and_signals(
    adata_umap: sc.AnnData,
    score_key: str,
    sig_key: str,
    *,
    trait_label: str = "IBD",
    cellpop_key: str = "Cell_population",
    notsig_label: str = "Not sig.",
    base_size: float = 6,
    causal_size: float = 20,
    signal_size: float = 16,
    other_alpha: float = 0.3,
    cmap: str = "RdBu_r",
    center_at: float = 0.0,
    composition_min_ratio: float = 0.01,
    composition_min_cells: int = 50,
    sort_descending: bool = False,
    savepath: Path | str | None = None,
    dpi: int = 300,
    show: bool = True,
):
    """Two-panel trait UMAP: scDRS-FM conditional disease scores and independent populations."""
    if "X_umap" not in adata_umap.obsm:
        raise ValueError("adata_umap.obsm['X_umap'] not found. Run prepare_umap first.")
    for required in [score_key, sig_key, cellpop_key]:
        if required not in adata_umap.obs:
            raise KeyError(f"{required!r} not found in adata_umap.obs")

    U = adata_umap.obsm["X_umap"]
    x, y = U[:, 0], U[:, 1]
    scores = pd.to_numeric(adata_umap.obs[score_key], errors="coerce").to_numpy()
    sig_raw = (
        pd.to_numeric(adata_umap.obs[sig_key].astype(str), errors="coerce")
        .fillna(-1)
        .astype(int)
        .to_numpy()
    )
    causal = sig_raw != -1
    cell_pops = adata_umap.obs[cellpop_key].astype(str).fillna("NA")
    total_by_celltype = cell_pops.value_counts()

    kept = sorted(np.unique(sig_raw[causal]).tolist())
    labels = np.array(
        [notsig_label if s == -1 else f"Indep. signal {s}" for s in sig_raw],
        dtype=object,
    )
    adata_umap.obs[f"{sig_key}_plot"] = pd.Categorical(
        labels,
        categories=[notsig_label] + [f"Indep. signal {s}" for s in kept],
        ordered=True,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    finite = np.isfinite(scores)
    max_abs = float(np.nanmax(np.abs(scores[finite] - center_at))) if finite.any() else 1.0
    if max_abs == 0:
        max_abs = 1.0
    norm = mcolors.TwoSlopeNorm(
        vcenter=center_at,
        vmin=center_at - max_abs,
        vmax=center_at + max_abs,
    )

    background_order = (
        descending_score_order(scores, ~causal)
        if sort_descending
        else np.flatnonzero(~causal)
    )
    causal_order = (
        descending_score_order(scores, causal)
        if sort_descending
        else np.flatnonzero(causal)
    )

    ax = axes[0]
    ax.scatter(
        x[background_order], y[background_order],
        c=scores[background_order], cmap=cmap, norm=norm,
        s=base_size, alpha=other_alpha, linewidths=0,
    )
    ax.scatter(
        x[causal_order], y[causal_order],
        c=scores[causal_order], cmap=cmap, norm=norm,
        s=causal_size, alpha=0.6, edgecolors="black", linewidths=0.4,
    )
    ax.set_title(f"scDRS-FM disease scores ({trait_label})", fontsize=18)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    ax2 = axes[1]
    notsig_color = "#D0D0D0"
    ax2.scatter(
        x[background_order], y[background_order],
        color=notsig_color, s=base_size, alpha=other_alpha, linewidths=0,
    )
    if len(kept) <= 10:
        palette = sns.color_palette("tab10", n_colors=len(kept)).as_hex()
    elif len(kept) <= 20:
        palette = sns.color_palette("tab20", n_colors=len(kept)).as_hex()
    else:
        palette = sns.color_palette("husl", n_colors=len(kept)).as_hex()
    sig_to_color = {s: palette[i] for i, s in enumerate(kept)}

    handles = [
        Line2D(
            [0], [0], marker="o", linestyle="None",
            markerfacecolor=notsig_color,
            markeredgecolor="black",  # legend-only black outline; plot background dots remain unoutlined
            markeredgewidth=0.8,
            markersize=7, alpha=other_alpha, label=notsig_label,
        )
    ]

    for s in kept:
        mask = sig_raw == s
        signal_order = (
            descending_score_order(scores, mask)
            if sort_descending
            else np.flatnonzero(mask)
        )
        ax2.scatter(
            x[signal_order], y[signal_order],
            color=sig_to_color[s], s=signal_size, alpha=0.6,
            edgecolors="black", linewidths=0.4,
        )

        signal_cell_counts = cell_pops[mask].value_counts()
        comp_rows = []
        for ct, in_signal in signal_cell_counts.items():
            total_count = int(total_by_celltype.get(ct, 0))
            if total_count == 0:
                continue
            ratio = in_signal / total_count
            if ratio > composition_min_ratio and in_signal >= composition_min_cells:
                comp_rows.append((ct, int(in_signal), total_count, ratio))
        comp_rows = sorted(comp_rows, key=lambda z: (-z[3], -z[1], z[0]))

        if comp_rows:
            comp_text = "\n".join(
                f"   {ct}: {in_signal}/{total_count}"
                for ct, in_signal, total_count, _ in comp_rows
            )
            label = f"Indep. population {s}\n{comp_text}"
        else:
            label = f"Indep. population {s}"

        handles.append(
            Line2D(
                [0], [0], marker="o", linestyle="None",
                markerfacecolor=sig_to_color[s], markeredgecolor="black",
                markersize=8, label=label,
            )
        )

    ax2.set_title(f"scDRS-FM independent populations ({trait_label})", fontsize=18)
    legend = ax2.legend(
        handles=handles, frameon=False, loc="center left",
        bbox_to_anchor=(1.02, 0.5), fontsize=12, handletextpad=0.8,
        labelspacing=1.2, borderaxespad=0.0,
    )
    for text_item in legend.get_texts():
        text_item.set_fontsize(12)
        text_item.set_multialignment("left")
        text_item.set_fontfamily("monospace")

    for axis in axes:
        clean_umap_axis(axis)

    if savepath is not None:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
        print(f"Saved {savepath}")
    if show:
        plt.show()
    else:
        plt.close(fig)


## Load datasets and build scDRS-FM tables

In [9]:
scores = pd.read_csv(RESULTS / 'ct' / 'soskic_immune_magic_ctrl' / 'UKB_460K.disease_AID_ALL.conditional.tagging_score.gz', sep='\t')
scores

,Unnamed: 0,cell_ids,metacell_size,independent_signal,independent_signal_multi,raw_score,norm_score,mc_pval,pval,nlog10_pval,zscore
0,0,"TCGCGTTAGTTTCCTT-1-122,TAGAGCTGTGGTCTCG-1-122,...",44,924,924,0.059692,0.561995,0.287712,0.281777,0.550095,0.577571
1,1,"TAGAGCTCAGACAAAT-1-120,GACACGCCATCTGGTA-1-105,...",44,924,924,-0.063185,-1.319937,0.942058,0.910540,0.040701,-1.344086
2,2,"GTGTGCGGTTAGTGGG-1-130,AATCCAGTCTTATCTG-1-21,C...",41,924,924,0.073296,0.563730,0.277722,0.281235,0.550930,0.579176
3,3,"TGTATTCAGCTAGGCA-1-114,TACACGAGTCTCAACA-1-36,C...",37,924,924,-0.111000,-1.074397,0.860140,0.861621,0.064684,-1.087632
4,4,"TCAGATGCACACATGT-1-130,GGGCATCAGAACTCGG-1-61,C...",36,924,924,0.113489,0.928982,0.171828,0.174669,0.757784,0.935874
...,...,...,...,...,...,...,...,...,...,...,...
929,929,"GGATGTTTCAAGAAGT-1-110,TTCTTAGGTTCCAACA-1-47,C...",5,70,70,0.102096,0.017753,0.524475,0.485725,0.313609,0.035789
930,930,"AGAGCTTCATGCATGT-1-46,TGGCGCATCAGTGCAT-1-46,AT...",5,70,70,0.077779,-0.720832,0.765235,0.764669,0.116526,-0.721404
931,931,"ACCGTAAAGCGATAGC-1-137,CCGGGATGTTGGTTTG-1-49,G...",5,924,924,-0.168878,-1.306279,0.905095,0.908215,0.041811,-1.329845
932,932,"CAGTAACTCGACAGCC-1-64,TACAGTGGTGTGCCTG-1-25,AC...",5,70,70,-0.162879,-0.012584,0.512488,0.498020,0.302753,0.004964


In [10]:
scores['nlog10_pval'].max()

3.9409635

In [11]:
analysis: dict[str, dict[str, object]] = {}

# Run only the first dataset
for dataset_name, config in list(DATASETS.items())[:1]:
    print(f"\n=== {config.label} ===")
    adata = load_and_preprocess_adata(config)

    tables = build_scdrsfm_celltype_tables(
        adata=adata,
        results_dir=config.scdrsfm_results_dir,
        traits=ALL_TRAITS,
        biocol=config.biocol,
        marginal_metacell_col=config.marginal_metacell_col,
        fdr_alpha=config.fdr_alpha,
        pval_col_candidates=("pval",),
        indep_sig_col=config.indep_sig_col,
        indep_cells_dir=Path("indep_cells") / dataset_name,
        heatmap_threshold=HEATMAP_THRESHOLD,
        heatmap_traits=HEATMAP_TRAITS,
        print_summaries=False,
    )
    analysis[dataset_name] = {"config": config, "adata": adata, **tables}

    # Save reusable tables.
    tables["df_marginal_props"].to_csv(TABLE_DIR / f"{dataset_name}_scdrsfm_marginal_props.tsv", sep="\t")
    tables["df_marginal_x_cond_props"].to_csv(TABLE_DIR / f"{dataset_name}_scdrsfm_marginal_x_cond_props.tsv", sep="\t")
    tables["df_indep_signal_counts"].to_csv(TABLE_DIR / f"{dataset_name}_scdrsfm_independent_signal_counts.tsv", sep="\t")

    print("Marginal cell-type proportions:")
    display(tables["df_marginal_props"])
    print("Marginal x conditional cell-type proportions:")
    display(tables["df_marginal_x_cond_props"])


=== Soskic ===


Soskic: loaded /mnt/shared-workspace/scdrsfm/data/subsets_10k/Soskic/soskic_100k.h5ad: 10,000 cells × 23,537 genes


Soskic: after filtering: 10,000 cells × 12,258 genes


Marginal cell-type proportions:


Cell_population,TN 0h,TN 40h,TN 16h,TCM 0h,TCM 5d,TCM 16h,TN 5d,TN HSP 5d,TCM 40h,TEM 0h,...,TEM HLA+ 40h,TEMRA 16h,TEMRA 40h,TEMRA 5d,TM cycling 5d,nTreg 0h,TN2 40h,TN IFN 16h,HSP 16h,TEMRA LA
trait,,,,,,,,,,,,,,,,,,,,,
PASS_CD_deLange2017,0.0,0.610548,0.992528,0.0,0.011401,0.989848,0.0,0.002747,0.610119,0.0,...,0.678571,0.291667,0.295455,0.022727,0.0,0.0,0.296296,1.00,1.00,0.0
PASS_Celiac,0.0,0.004057,0.052304,0.0,0.001629,0.138748,0.0,0.000000,0.002976,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.08,0.60,0.0
PASS_IBD_deLange2017,0.0,0.597363,0.988792,0.0,0.003257,0.984772,0.0,0.002747,0.419643,0.0,...,0.571429,0.187500,0.204545,0.022727,0.0,0.0,0.259259,1.00,1.00,0.0
PASS_Lupus,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.00,0.0
PASS_Multiple_sclerosis,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.00,0.0
PASS_Primary_biliary_cirrhosis,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.00,0.0
PASS_Rheumatoid_Arthritis,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.00,0.0
PASS_Type_1_Diabetes,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.00,0.00,0.0
PASS_UC_deLange2017,0.0,0.128803,0.630137,0.0,0.001629,0.915398,0.0,0.000000,0.071429,0.0,...,0.125000,0.104167,0.022727,0.000000,0.0,0.0,0.222222,0.64,0.96,0.0


Marginal x conditional cell-type proportions:


Cell_population,TN 0h,TN 40h,TN 16h,TCM 0h,TCM 5d,TCM 16h,TN 5d,TN HSP 5d,TCM 40h,TEM 0h,...,TEM HLA+ 40h,TEMRA 16h,TEMRA 40h,TEMRA 5d,TM cycling 5d,nTreg 0h,TN2 40h,TN IFN 16h,HSP 16h,TEMRA LA
trait,,,,,,,,,,,,,,,,,,,,,
PASS_CD_deLange2017,0.0,0.003043,0.021171,0.0,0.0,0.064298,0.0,0.0,0.002976,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_Celiac,0.0,0.003043,0.043587,0.0,0.0,0.121827,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.04,0.6,0.0
PASS_IBD_deLange2017,0.0,0.002028,0.009963,0.0,0.0,0.069374,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_Lupus,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_Multiple_sclerosis,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_Primary_biliary_cirrhosis,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_Rheumatoid_Arthritis,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_Type_1_Diabetes,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0
PASS_UC_deLange2017,0.0,0.056795,0.368618,0.0,0.0,0.751269,0.0,0.0,0.029762,0.0,...,0.053571,0.020833,0.0,0.0,0.0,0.0,0.148148,0.52,0.8,0.0


## scDRS-FM heatmaps: Soskic primary and Nathan / Cano-Gamez validation

In [12]:
for dataset_name in ["soskic"]:#, "nathan", "canogamez"]:
    item = analysis[dataset_name]
    config: DatasetConfig = item["config"]
    print(f"\n=== Plotting {config.label} scDRS-FM heatmap ===")
    plot_scdrsfm_heatmap(
        adata=item["adata"],
        biocol=config.biocol,
        df_marginal_props=item["df_marginal_props"],
        df_marginal_x_cond_props=item["df_marginal_x_cond_props"],
        df_signal_details=item["df_signal_details"],
        trait_order=HEATMAP_TRAITS,
        trait_labels=TRAIT_LABELS,
        marginal_threshold=HEATMAP_THRESHOLD,
        conditional_threshold=HEATMAP_THRESHOLD,
        signal_cell_ids_col="marg_x_signal_cell_ids_cond_sig",
        title=f"{config.label}: scDRS-FM",
        out_png=HEATMAP_DIR / f"{dataset_name}_scdrsfm_all_marginal_celltypes.png",
        out_csv=(SOSKIC_SCDRSFM_CSV if dataset_name == "soskic" else MANUSCRIPT_SUPPLEMENTARY_DIR / f"{dataset_name}_scDRSFM_all_marginal_celltypes_cell_type_proportions.csv"),
        fontsize_mult=1.1,
    )


=== Plotting Soskic scDRS-FM heatmap ===


Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/Soskic_scDRSFM_all_marginal_celltypes_cell_type_proportions.csv (190 rows)
Saved t_cell_analysis_outputs/heatmaps/soskic_scdrsfm_all_marginal_celltypes.png


## Soskic scDRS heatmap

In [13]:
# === scPagwas guard (injected, P5): skip scPagwas cleanly when its data is absent ===
# User decision (2026-07-18): guard scPagwas reader fns to return empty + auto-skip
# the scPagwas-only plots.  scDRS / scDRS-FM logic and figures are untouched.
import functools as _functools

_SCPAGWAS_PRESENT = bool(SCPAGWAS_SOSKIC_DIR) and _Path(SCPAGWAS_SOSKIC_DIR).exists()
print(f"[guard] scPagwas data present: {_SCPAGWAS_PRESENT} ({SCPAGWAS_SOSKIC_DIR})")


def _empty_sig_pair(cells):
    import numpy as _np
    s = pd.Series(_np.nan, index=pd.Index(cells, dtype=str), dtype=float)
    m = pd.Series(False, index=pd.Index(cells, dtype=str), dtype=bool)
    s.attrs["scpagwas_absent"] = True
    return s, m


if not _SCPAGWAS_PRESENT:
    # 1. reader used inside the per-trait UMAP loop -> empty (blank panel, no crash)
    def read_scpagwas_score_and_sig(score_file, cells, *, score_col="scPagwas.TRS.Score",
                                    sig_col="Random_Correct_BG_adjp", sig_alpha=0.1):
        return _empty_sig_pair(cells)

    # 2. table builder used by the scPagwas heatmap cell -> empty, sentinel-tagged
    def build_scpagwas_tables(*, adata, base_dir, traits, biocol="Cell_population",
                              cell_alpha=0.1, ct_alpha=0.05,
                              score_sig_col="Random_Correct_BG_adjp", ct_pval_col="pvalue"):
        df_props = pd.DataFrame(index=pd.Index([], name="trait"))
        df_props.attrs["scpagwas_absent"] = True
        df_assoc = pd.DataFrame(index=pd.Index([], name="trait"))
        df_assoc.attrs["scpagwas_absent"] = True
        return df_props, df_assoc

    # 3. shared heatmap plotter -> early no-op ONLY for empty/sentinel scPagwas data
    _orig_plot_ct_assoc = plot_celltype_association_heatmap

    @_functools.wraps(_orig_plot_ct_assoc)
    def plot_celltype_association_heatmap(*args, **kwargs):
        prop = kwargs.get("prop_df")
        if prop is None and args:
            prop = args[0]
        if isinstance(prop, pd.DataFrame) and (prop.attrs.get("scpagwas_absent") or prop.empty):
            print("[guard] scPagwas heatmap skipped (no scPagwas data).")
            return None, None
        return _orig_plot_ct_assoc(*args, **kwargs)


[guard] scPagwas data present: False (/mnt/shared-workspace/scdrsfm/results/scpagwas/soskic)


In [14]:
soskic_item = analysis["soskic"]
soskic_config: DatasetConfig = soskic_item["config"]
soskic_adata: sc.AnnData = soskic_item["adata"]

scdrs_marginal_props = build_marginal_props_from_score_files(
    adata=soskic_adata,
    results_dir=SCDRS_SOSKIC_RESULTS,
    traits=ALL_TRAITS,
    biocol=soskic_config.biocol,
    fdr_alpha=0.1,
    pval_col_candidates=("pval", "mc_pval"),
)

scdrs_celltype_assoc = build_scdrs_celltype_association_matrix(
    results_dir=SCDRS_SOSKIC_RESULTS,
    traits=ALL_TRAITS,
    biocol=soskic_config.biocol,
    alpha=0.05,
    pval_col="assoc_mcp",
)

scdrs_marginal_props.to_csv(TABLE_DIR / "soskic_scdrs_marginal_props.tsv", sep="\t")
scdrs_celltype_assoc.to_csv(TABLE_DIR / "soskic_scdrs_celltype_assoc.tsv", sep="\t")

plot_celltype_association_heatmap(
    adata=soskic_adata,
    biocol=soskic_config.biocol,
    prop_df=scdrs_marginal_props,
    assoc_df=scdrs_celltype_assoc,
    trait_order=HEATMAP_TRAITS,
    trait_labels=TRAIT_LABELS,
    title="Soskic: scDRS",
    colorbar_label="Prop. sig. marginal cells",
    star_label="scDRS cell-type association",
    out_png=HEATMAP_DIR / "soskic_scdrs_celltype_assoc_heatmap.png",
    out_csv=SOSKIC_SCDRS_CSV,
    fontsize_mult=1.1,
)

Saved heatmap cell-type proportions: nature_genetics_manuscript_supplementary/Soskic_scDRS_celltype_assoc_heatmap_cell_type_proportions.csv (80 rows)
Saved t_cell_analysis_outputs/heatmaps/soskic_scdrs_celltype_assoc_heatmap.png


(Cell_population                            HSP 16h   TCM 16h   TEM 16h  \
 trait                                                                    
 PASS_CD_deLange2017                           0.00  0.010152  0.053004   
 PASS_UC_deLange2017                           0.00  0.005076  0.014134   
 PASS_IBD_deLange2017                          0.04  0.015228  0.060071   
 PASS_Celiac                                   0.04  0.025381  0.014134   
 PASS_Rheumatoid_Arthritis                     0.00  0.001692  0.000000   
 UKB_460K.disease_AID_ALL                      0.00  0.000000  0.000000   
 UKB_460K.disease_ASTHMA_DIAGNOSED             0.04  0.025381  0.042403   
 UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED     0.00  0.027073  0.014134   
 UKB_460K.disease_RESPIRATORY_ENT              0.04  0.018613  0.045936   
 UKB_460K.disease_HYPOTHYROIDISM_SELF_REP      0.00  0.000000  0.000000   
 
 Cell_population                              TN 16h  TN IFN 16h   TN NFKB  \
 trait              

## Soskic scPagwas heatmap

In [15]:
scp_props, scp_celltype_assoc = build_scpagwas_tables(
    adata=soskic_adata,
    base_dir=SCPAGWAS_SOSKIC_DIR,
    traits=ALL_TRAITS,
    biocol=soskic_config.biocol,
    cell_alpha=0.1,
    ct_alpha=0.05,
    score_sig_col="Random_Correct_BG_adjp",
    ct_pval_col="pvalue",
)

scp_props.to_csv(TABLE_DIR / "soskic_scpagwas_sig_cell_props.tsv", sep="\t")
scp_celltype_assoc.to_csv(TABLE_DIR / "soskic_scpagwas_celltype_assoc.tsv", sep="\t")

plot_celltype_association_heatmap(
    adata=soskic_adata,
    biocol=soskic_config.biocol,
    prop_df=scp_props,
    assoc_df=scp_celltype_assoc,
    trait_order=HEATMAP_TRAITS,
    trait_labels=TRAIT_LABELS,
    title="Soskic: scPagwas",
    colorbar_label="Prop. sig. scPagwas cells",
    star_label="scPagwas cell-type association",
    out_png=HEATMAP_DIR / "soskic_scpagwas_celltype_assoc_heatmap.png",
    out_csv=SOSKIC_SCPAGWAS_CSV,
    fontsize_mult=1.1,
)

[guard] scPagwas heatmap skipped (no scPagwas data).


(None, None)

## Soskic CD4 T-cell population UMAP

In [16]:

# Prepare the Soskic UMAP once and reuse it for all downstream UMAP panels.
adata_umap = prepare_umap(soskic_adata, force=False)

plot_cell_population_umap(
    adata_umap,
    key=soskic_config.biocol,
    title="CD4 T cells",
    out_png=UMAP_DIR / "soskic_cd4_t_cell_populations_umap.png",
    label_fontsize=10,
)


/workspace/scdrsfm_env/lib/python3.11/site-packages/scanpy/preprocessing/_pca.py:374: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)


Saved t_cell_analysis_outputs/umaps/soskic_cd4_t_cell_populations_umap.png


## Soskic scDRS-FM conditional disease scores and independent populations for all traits

In [17]:
# scDRS-FM conditional disease-score and independent-population UMAPs for every trait.
# IBD is still displayed inline and retained through compatibility aliases for downstream DE/pathway code.
scdrsfm_conditional_umap_results: dict[str, dict[str, object]] = {}
df_cond_ibd = None
df_marg_ibd = None
ibd_signal_remap = None

for trait in UMAP_TRAITS:
    trait_prefix = Path(trait).name
    trait_label = TRAIT_LABELS.get(trait, trait_prefix)

    df_cond_trait, df_marg_trait, signal_remap = assign_conditional_scores_to_cells_filtered_signals(
        adata_umap=adata_umap,
        out_folder=soskic_config.scdrsfm_results_dir,
        trait=trait,
        indep_sig_col=soskic_config.indep_sig_col,
        fdr_alpha=soskic_config.fdr_alpha,
        min_causal_cells_per_signal=100,
        cellpop_key=soskic_config.biocol,
        min_fraction_within_any_cellpop=0.05,
    )

    trait_score_key = f"{trait_prefix}_conditional_score"
    trait_sig_key = f"{trait_prefix}_{soskic_config.indep_sig_col}_filtered"
    scdrsfm_conditional_umap_results[trait] = {
        "score_key": trait_score_key,
        "sig_key": trait_sig_key,
        "signal_remap": signal_remap,
    }

    if trait == UMAP_TRAIT:
        out_png = UMAP_DIR / "soskic_ibd_scdrsfm_scores_and_independent_populations.png"
        df_cond_ibd = df_cond_trait
        df_marg_ibd = df_marg_trait
        ibd_signal_remap = signal_remap
    else:
        out_png = UMAP_DIR / f"soskic_{trait_prefix}_scdrsfm_scores_and_independent_populations.png"

    plot_ibd_scores_and_signals(
        adata_umap,
        trait_score_key,
        trait_sig_key,
        trait_label=trait_label,
        cellpop_key=soskic_config.biocol,
        composition_min_ratio=0.01,
        composition_min_cells=50,
        other_alpha=0.3,
        signal_size=16,
        sort_descending=False,
        savepath=out_png,
        show=(trait == UMAP_TRAIT),
    )

if df_cond_ibd is None or df_marg_ibd is None or ibd_signal_remap is None:
    raise RuntimeError(f"Focal IBD trait {UMAP_TRAIT!r} was not included in UMAP_TRAITS")

ibd_score_key = scdrsfm_conditional_umap_results[UMAP_TRAIT]["score_key"]
ibd_sig_key = scdrsfm_conditional_umap_results[UMAP_TRAIT]["sig_key"]

# Compatibility aliases for the restored final heatmap logic.
score_key = ibd_score_key
sig_key = ibd_sig_key


Saved t_cell_analysis_outputs/umaps/soskic_PASS_CD_deLange2017_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_PASS_Celiac_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_ibd_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_PASS_Lupus_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_PASS_Multiple_sclerosis_scdrsfm_scores_and_independent_populations.png


...

Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_AID_ALL_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_HYPOTHYROIDISM_SELF_REP_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_RESPIRATORY_ENT_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED_scdrsfm_scores_and_independent_populations.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_ASTHMA_DIAGNOSED_scdrsfm_scores_and_independent_populations.png


## Soskic phenotype-score UMAPs for all functional phenotypes

In [18]:
# Generate and save a UMAP for every functional phenotype score.
missing_phenotype_score_files = [
    PHENO_SOSKIC_RESULTS / f"{phenotype}.marginal_score.gz"
    for phenotype in PHENOTYPE_TRAITS
    if not (PHENO_SOSKIC_RESULTS / f"{phenotype}.marginal_score.gz").exists()
]
if missing_phenotype_score_files:
    missing_text = "\n".join(str(path) for path in missing_phenotype_score_files)
    raise FileNotFoundError(f"Missing phenotype score files:\n{missing_text}")

for phenotype in PHENOTYPE_TRAITS:
    score_file = PHENO_SOSKIC_RESULTS / f"{phenotype}.marginal_score.gz"
    scores = read_tsv_score_series(score_file, adata_umap.obs_names, score_col="norm_score")
    plot_one_score_umap(
        adata_umap,
        scores,
        title=f"{phenotype} phenotype scores",
        out_png=UMAP_DIR / f"soskic_{phenotype.replace('-', '_').replace(' ', '_')}_phenotype_scores_umap.png",
        sig_mask=None,
        cmap="RdBu_r",
        center_at=0.0,
        base_size=8,
        base_alpha=1.0,
        sort_descending=False,
        show=phenotype in {"Multi-Cytokine", "CTLA4-CD38"},
    )


Saved t_cell_analysis_outputs/umaps/soskic_Metallothionein_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_Translation_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_IL10_IL19_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_OX40_EBI3_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_CD172a_MERTK_phenotype_scores_umap.png


...

Saved t_cell_analysis_outputs/umaps/soskic_IEG2_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_Cytoskeleton_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_CTLA4_CD38_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_Multi_Cytokine_phenotype_scores_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_ICOS_CD38_phenotype_scores_umap.png


## Soskic scDRS-FM / scDRS / scPagwas disease-score UMAPs for all traits

In [19]:
# Disease-score UMAPs for every trait and every method.
# Score-based descending point ordering is disabled for every panel.
disease_umap_files: list[Path] = []

for trait in UMAP_TRAITS:
    trait_prefix = Path(trait).name
    trait_label = TRAIT_LABELS.get(trait, trait_prefix)

    scdrsfm_scores, scdrsfm_sig = read_scdrs_like_score_and_sig(
        soskic_config.scdrsfm_results_dir / f"{trait_prefix}.marginal_score.gz",
        adata_umap.obs_names,
        score_col="norm_score",
        pval_col_candidates=("pval", "mc_pval"),
        fdr_alpha=soskic_config.fdr_alpha,
    )

    scdrs_scores, scdrs_sig = read_scdrs_like_score_and_sig(
        SCDRS_SOSKIC_RESULTS / f"{trait_prefix}.marginal_score.gz",
        adata_umap.obs_names,
        score_col="norm_score",
        pval_col_candidates=("pval", "mc_pval"),
        fdr_alpha=0.1,
    )

    scp_scores, scp_sig = read_scpagwas_score_and_sig(
        scpagwas_singlecell_file(SCPAGWAS_SOSKIC_DIR, trait),
        adata_umap.obs_names,
        score_col="scPagwas.TRS.Score",
        sig_col="Random_Correct_BG_adjp",
        sig_alpha=0.1,
    )

    method_data = [
        ("scdrsfm", "scDRS-FM marginal disease scores", scdrsfm_scores, scdrsfm_sig),
        ("scdrs", "scDRS disease scores", scdrs_scores, scdrs_sig),
        ("scpagwas", "scPagwas disease scores", scp_scores, scp_sig),
    ]

    # Save every method-specific panel as its own figure.
    for method_slug, method_title, method_scores, method_sig in method_data:
        method_out = UMAP_DIR / f"soskic_{trait_prefix}_{method_slug}_disease_scores_highlighted_umap.png"
        plot_one_score_umap(
            adata_umap,
            method_scores,
            title=f"{method_title}\n{trait_label}",
            out_png=method_out,
            sig_mask=method_sig,
            cmap="RdBu_r",
            center_at=0.0,
            base_size=6,
            sig_size=22,
            base_alpha=0.35,
            sig_alpha=0.85,
            sort_descending=False,
            show=False,
        )
        disease_umap_files.append(method_out)

    # Preserve the original combined scDRS-FM / scDRS / scPagwas comparison layout.
    combined_out = UMAP_DIR / f"{trait_prefix}_scdrsfm_scdrs_scpagwas_disease_scores_highlighted.png"
    plot_score_umap_panels_highlight(
        adata_umap,
        {
            f"scDRS-FM marginal disease scores\n{trait_label}": (scdrsfm_scores, scdrsfm_sig),
            f"scDRS disease scores\n{trait_label}": (scdrs_scores, scdrs_sig),
            f"scPagwas disease scores\n{trait_label}": (scp_scores, scp_sig),
        },
        out_png=combined_out,
        cmap="RdBu_r",
        center_at=0.0,
        base_size=6,
        sig_size=22,
        base_alpha=0.35,
        sig_alpha=0.85,
        sort_descending=False,
        show=(trait == UMAP_TRAIT),
    )
    disease_umap_files.append(combined_out)

print(f"Saved {len(disease_umap_files)} marginal disease-score UMAP figures.")


Saved t_cell_analysis_outputs/umaps/soskic_PASS_CD_deLange2017_scdrsfm_disease_scores_highlighted_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_PASS_CD_deLange2017_scdrs_disease_scores_highlighted_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_PASS_CD_deLange2017_scpagwas_disease_scores_highlighted_umap.png


Saved t_cell_analysis_outputs/umaps/PASS_CD_deLange2017_scdrsfm_scdrs_scpagwas_disease_scores_highlighted.png


Saved t_cell_analysis_outputs/umaps/soskic_PASS_Celiac_scdrsfm_disease_scores_highlighted_umap.png


...

Saved t_cell_analysis_outputs/umaps/UKB_460K.disease_ALLERGY_ECZEMA_DIAGNOSED_scdrsfm_scdrs_scpagwas_disease_scores_highlighted.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_ASTHMA_DIAGNOSED_scdrsfm_disease_scores_highlighted_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_ASTHMA_DIAGNOSED_scdrs_disease_scores_highlighted_umap.png


Saved t_cell_analysis_outputs/umaps/soskic_UKB_460K.disease_ASTHMA_DIAGNOSED_scpagwas_disease_scores_highlighted_umap.png


Saved t_cell_analysis_outputs/umaps/UKB_460K.disease_ASTHMA_DIAGNOSED_scdrsfm_scdrs_scpagwas_disease_scores_highlighted.png
Saved 56 marginal disease-score UMAP figures.


## Functional phenotype gene sets

In [20]:

import scipy.sparse as sp
from scipy.stats import mannwhitneyu, fisher_exact

functional_phenotypes = list(PHENOTYPE_TRAITS)


def read_geneset_file(path: Path) -> tuple[str, list[str]]:
    df = pd.read_csv(path, sep="\t")
    if not {"TRAIT", "GENESET"}.issubset(df.columns):
        raise ValueError(f"{path} missing required columns TRAIT and GENESET. Found: {df.columns.tolist()}")
    trait = str(df.loc[0, "TRAIT"])
    genes = [gene.strip() for gene in str(df.loc[0, "GENESET"]).split(",") if str(gene).strip()]
    return trait, genes


def load_functional_genesets(phenotypes: list[str], folder: Path):
    rows_long = []
    rows_wide = []
    genesets = {}

    for phenotype in phenotypes:
        candidates = sorted(Path(folder).glob(f"{phenotype}*"))
        if not candidates:
            raise FileNotFoundError(f"No file found for phenotype {phenotype!r} in {folder}")
        path = candidates[0]
        trait, genes = read_geneset_file(path)
        genesets[phenotype] = genes
        rows_wide.append({
            "phenotype": phenotype,
            "trait_in_file": trait,
            "file": path.name,
            "n_genes_listed": len(genes),
            "geneset": ",".join(genes),
        })
        rows_long.extend({"phenotype": phenotype, "file": path.name, "gene": gene} for gene in genes)

    return genesets, pd.DataFrame(rows_long), pd.DataFrame(rows_wide)


genesets, geneset_long_df, geneset_wide_df = load_functional_genesets(functional_phenotypes, T_CELL_PHENO_DIR)
print("Functional phenotype gene sets:")
display(geneset_wide_df.head())


Functional phenotype gene sets:


,phenotype,trait_in_file,file,n_genes_listed,geneset
0,Metallothionein,Metallothionein,Metallothionein,200,"MT1X,MT2A,MT1E,MT1G,MT1H,MT1F,MT1M,CCNA1,SLC30..."
1,Translation,Translation,Translation,200,"RPS18,RPS6,RPS12,RPL3,RPS3,RPS4X,RPL13,RPS3A,E..."
2,IL10-IL19,IL10-IL19,IL10-IL19,200,"CAV1,PTPN3,FXYD2,AB_CD38-2,IL19,MYB,IL10,TIMD4..."
3,OX40-EBI3,OX40-EBI3,OX40-EBI3,200,"EBI3,TNFRSF4,TNFRSF18,PRKCDBP,PKM,CCL22,PTP4A3..."
4,CD172a-MERTK,CD172a-MERTK,CD172a-MERTK,200,"TXNIP,AB_CD172a,AB_MERTK,AB_CD138-1,AB_CD324,C..."


## Differential-expression analyses for the final pathway heatmap

In [21]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

# ============================================================
# Parameters
# ============================================================
celltype_col = "Cell_population"

alpha = 0.05
lfc_min = 0.25           # Set to 0.0 to disable logFC filtering
use_raw = False
layer = None             # Set if expression is stored in a layer

# ============================================================
# Explicitly rebuild the focal IBD independent populations
# ============================================================
# This prevents `sig_key` from referring to a stale or non-IBD
# independent-population column.
trait_prefix = Path(UMAP_TRAIT).name

df_cond_ibd, df_marg_ibd, ibd_signal_remap = (
    assign_conditional_scores_to_cells_filtered_signals(
        adata_umap=adata_umap,
        out_folder=soskic_config.scdrsfm_results_dir,
        trait=UMAP_TRAIT,
        indep_sig_col=soskic_config.indep_sig_col,
        fdr_alpha=soskic_config.fdr_alpha,
        min_causal_cells_per_signal=100,
        cellpop_key=soskic_config.biocol,
        min_fraction_within_any_cellpop=0.05,
    )
)

ibd_score_key = f"{trait_prefix}_conditional_score"
ibd_sig_key = (
    f"{trait_prefix}_"
    f"{soskic_config.indep_sig_col}_filtered"
)

# Preserve the aliases used by later notebook cells.
score_key = ibd_score_key
sig_key = ibd_sig_key
signal_col = ibd_sig_key

if signal_col not in adata_umap.obs.columns:
    raise KeyError(
        f"Expected IBD signal column {signal_col!r} was not created."
    )

if celltype_col not in adata_umap.obs.columns:
    raise KeyError(
        f"Expected cell-type column {celltype_col!r} was not found."
    )

# ============================================================
# Background definitions for this dataset
# ============================================================
signal1_background = [
    "TN NFKB",
]

signal2_background = [
    "TN NFKB",
    "TEM 16h",
    "TCM 16h",
    "TN 16h",
]

# ============================================================
# Helper
# ============================================================
def run_target_vs_all(
    adata,
    target_mask,
    comparison_name,
    alpha=0.05,
    lfc_min=0.25,
    use_raw=False,
    layer=None,
):
    """
    Run differential expression for target cells versus every
    other cell in the supplied AnnData object.
    """
    ad = adata.copy()

    if isinstance(target_mask, pd.Series):
        target_mask = (
            target_mask
            .reindex(ad.obs_names, fill_value=False)
            .fillna(False)
            .astype(bool)
        )
    else:
        target_mask = pd.Series(
            np.asarray(target_mask),
            index=ad.obs_names,
        ).fillna(False).astype(bool)

    n_target = int(target_mask.sum())
    n_rest = int((~target_mask).sum())

    if n_target == 0:
        raise ValueError(
            f"{comparison_name}: no target cells found"
        )

    if n_rest == 0:
        raise ValueError(
            f"{comparison_name}: no background/rest cells available"
        )

    target_label = "Target"
    rest_label = "Rest"
    group_key = f"deg_group_{comparison_name}"

    ad.obs[group_key] = pd.Categorical(
        np.where(
            target_mask.reindex(ad.obs_names).to_numpy(),
            target_label,
            rest_label,
        ),
        categories=[target_label, rest_label],
        ordered=True,
    )

    sc.tl.rank_genes_groups(
        ad,
        groupby=group_key,
        groups=[target_label],
        reference=rest_label,
        method="wilcoxon",
        use_raw=use_raw,
        layer=layer,
        n_genes=ad.n_vars,
    )

    df = sc.get.rank_genes_groups_df(
        ad,
        group=target_label,
    )

    keep = (
        df["pvals_adj"].notna()
        & (df["pvals_adj"] < alpha)
    )

    if (
        "logfoldchanges" in df.columns
        and df["logfoldchanges"].notna().any()
    ):
        keep &= df["logfoldchanges"].abs() >= lfc_min

    df_sig = df.loc[keep].copy()
    df_sig["comparison"] = comparison_name
    df_sig["n_target_cells"] = n_target
    df_sig["n_rest_cells"] = n_rest

    return {
        "adata_subset": ad,
        "all_ranked_genes": df,
        "sig_degs": df_sig,
        "deg_set": set(df_sig["names"].astype(str)),
        "n_target_cells": n_target,
        "n_rest_cells": n_rest,
    }


# ============================================================
# Precompute masks
# ============================================================
sig_raw = (
    pd.to_numeric(
        adata_umap.obs[signal_col].astype("string"),
        errors="coerce",
    )
    .fillna(-1)
    .astype(int)
)

# Do not rely on a stale assumption about which signal IDs exist.
# The assignment function remaps retained populations to 1, 2, ...
available_signal_ids = sorted(
    int(signal_id)
    for signal_id in pd.unique(sig_raw)
    if int(signal_id) > 0
)

signal_counts = (
    sig_raw
    .value_counts()
    .sort_index()
    .to_dict()
)

print(f"IBD signal column: {signal_col}")
print(f"IBD signal remapping: {ibd_signal_remap}")
print(f"IBD signal counts: {signal_counts}")

# NOTE (10k-scale reproduction edit): the original notebook required >=2 retained
# IBD independent populations. At the 10k-cell subsample only one IBD population
# survives the concentration filter, so we relax the guard to require >=1 and, when
# only one population is present, reuse it for the "signal 2" slot so the downstream
# 4-row DEG heatmap still renders (its two "signal" rows become identical, which
# transparently reflects that a single independent population was recovered). The
# background rows use cell-type-based backgrounds and remain distinct. scDRS-FM
# package code is unchanged; this edit only affects notebook plotting logic.
if len(available_signal_ids) < 1:
    raise ValueError(
        "No retained IBD independent populations were found. "
        f"Available positive signal IDs: {available_signal_ids}. "
        f"Full signal counts: {signal_counts}."
    )

signal1_id = available_signal_ids[0]
if len(available_signal_ids) >= 2:
    signal2_id = available_signal_ids[1]
else:
    signal2_id = signal1_id
    print(
        f"[10k-scale note] Only one retained IBD independent population "
        f"(ID {signal1_id}); reusing it for the signal-2 slot."
    )

signal1_mask = sig_raw.eq(signal1_id)
signal2_mask = sig_raw.eq(signal2_id)

celltype_vals = (
    adata_umap.obs[celltype_col]
    .astype("string")
)

# Original notebook procedure: use every cell in each selected
# background cell population, including any independent-signal cells.
signal1_background_mask = celltype_vals.isin(signal1_background)
signal2_background_mask = celltype_vals.isin(signal2_background)

print(
    f"Signal 1 (ID {signal1_id}) target cells: "
    f"{int(signal1_mask.sum()):,}"
)
print(
    f"Signal 2 (ID {signal2_id}) target cells: "
    f"{int(signal2_mask.sum()):,}"
)
print(
    "Signal 1 background cells: "
    f"{int(signal1_background_mask.sum()):,}"
)
print(
    "Signal 2 background cells: "
    f"{int(signal2_background_mask.sum()):,}"
)

# ============================================================
# Run the 4 DEG analyses
# ============================================================
deg_results = {}

# 1) Signal 1 vs all
print("\nRunning signal 1 vs all")
deg_results["signal1_vs_all"] = run_target_vs_all(
    adata=adata_umap,
    target_mask=signal1_mask,
    comparison_name="signal1_vs_all",
    alpha=alpha,
    lfc_min=lfc_min,
    use_raw=use_raw,
    layer=layer,
)

# 2) Signal 2 vs all
print("Running signal 2 vs all")
deg_results["signal2_vs_all"] = run_target_vs_all(
    adata=adata_umap,
    target_mask=signal2_mask,
    comparison_name="signal2_vs_all",
    alpha=alpha,
    lfc_min=lfc_min,
    use_raw=use_raw,
    layer=layer,
)

# 3) Signal 1 background vs all
print("Running signal 1 background vs all")
deg_results["signal1_background_vs_all"] = run_target_vs_all(
    adata=adata_umap,
    target_mask=signal1_background_mask,
    comparison_name="signal1_background_vs_all",
    alpha=alpha,
    lfc_min=lfc_min,
    use_raw=use_raw,
    layer=layer,
)

# 4) Signal 2 background vs all
print("Running signal 2 background vs all")
deg_results["signal2_background_vs_all"] = run_target_vs_all(
    adata=adata_umap,
    target_mask=signal2_background_mask,
    comparison_name="signal2_background_vs_all",
    alpha=alpha,
    lfc_min=lfc_min,
    use_raw=use_raw,
    layer=layer,
)

# ============================================================
# Collect outputs
# ============================================================
deg_tables = []
deg_sets = {}

for name, result in deg_results.items():
    deg_tables.append(result["sig_degs"].copy())
    deg_sets[name] = result["deg_set"]

deg_df = (
    pd.concat(deg_tables, ignore_index=True)
    if deg_tables
    else pd.DataFrame()
)

# ============================================================
# Summary / sanity check
# ============================================================
print()

for name, result in deg_results.items():
    print(
        f"{name}: "
        f"{result['n_target_cells']:,} target cells vs "
        f"{result['n_rest_cells']:,} rest cells | "
        f"{len(result['sig_degs']):,} significant DEGs"
    )

display(deg_df.head())

# Useful accessors:
# deg_results["signal1_vs_all"]["all_ranked_genes"]
# deg_results["signal1_vs_all"]["sig_degs"]
# deg_results["signal2_vs_all"]["all_ranked_genes"]
# deg_results["signal2_vs_all"]["sig_degs"]
# deg_results["signal1_background_vs_all"]["all_ranked_genes"]
# deg_results["signal1_background_vs_all"]["sig_degs"]
# deg_results["signal2_background_vs_all"]["all_ranked_genes"]
# deg_results["signal2_background_vs_all"]["sig_degs"]
# deg_sets["signal1_vs_all"]
# deg_sets["signal2_vs_all"]
# deg_sets["signal1_background_vs_all"]
# deg_sets["signal2_background_vs_all"]

IBD signal column: PASS_IBD_deLange2017_independent_signal_filtered
IBD signal remapping: {274: 1}
IBD signal counts: {-1: 9886, 1: 114}
[10k-scale note] Only one retained IBD independent population (ID 1); reusing it for the signal-2 slot.
Signal 1 (ID 1) target cells: 114
Signal 2 (ID 1) target cells: 114
Signal 1 background cells: 106
Signal 2 background cells: 1,783

Running signal 1 vs all


Running signal 2 vs all


Running signal 1 background vs all


Running signal 2 background vs all



signal1_vs_all: 114 target cells vs 9,886 rest cells | 1,801 significant DEGs
signal2_vs_all: 114 target cells vs 9,886 rest cells | 1,801 significant DEGs
signal1_background_vs_all: 106 target cells vs 9,894 rest cells | 2,071 significant DEGs
signal2_background_vs_all: 1,783 target cells vs 8,217 rest cells | 5,046 significant DEGs


,names,scores,logfoldchanges,pvals,pvals_adj,comparison,n_target_cells,n_rest_cells
0,MIR155HG,14.715924,3.491456,5.094555e-49,3.243725e-45,signal1_vs_all,114,9886
1,NAMPT,14.547558,3.472155,6.052584e-48,2.473086e-44,signal1_vs_all,114,9886
2,TNFRSF4,13.826012,2.807249,1.775970e-43,5.442460e-40,signal1_vs_all,114,9886
3,DDX21,13.537929,2.673430,9.338471e-42,2.289420e-38,signal1_vs_all,114,9886
4,VSIR,13.501303,3.588776,1.536352e-41,3.138767e-38,signal1_vs_all,114,9886


## Enrichr / phenotype gene-set setup

In [22]:
import numpy as np
import pandas as pd

def _upper_set(xs):
    return set(pd.Index(list(xs)).astype(str).str.upper())

def parse_geneset_string(x):
    if pd.isna(x):
        return set()
    return {
        g.strip().upper()
        for g in str(x).split(",")
        if str(g).strip() != ""
    }

# Your existing phenotype gene sets from geneset_wide_df
PHENOTYPE_GENESETS = (
    geneset_wide_df.groupby("phenotype")["geneset"]
    .apply(lambda s: set().union(*[parse_geneset_string(x) for x in s]))
    .to_dict()
)

# Gene universe
GENE_UNIVERSE = _upper_set(soskic_adata.var_names)

In [23]:
# If needed:
# !pip install gseapy

import re
import pickle
from pathlib import Path
import pandas as pd
import gseapy as gp

CACHE_DIR = Path("./geneset_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def canonicalize_geneset_dict(gs_dict):
    out = {}
    for term, genes in gs_dict.items():
        clean = sorted({
            str(g).strip().upper()
            for g in genes
            if pd.notna(g) and str(g).strip() != ""
        })
        if len(clean) > 0:
            out[str(term)] = clean
    return out

def extract_year(name):
    m = re.search(r"(\d{4})$", name)
    return int(m.group(1)) if m else -1

def latest_matching_library(available_libs, prefix):
    matches = [x for x in available_libs if x.startswith(prefix)]
    if not matches:
        raise ValueError(f"No Enrichr library found matching prefix: {prefix}")
    matches = sorted(matches, key=lambda x: (extract_year(x), x))
    return matches[-1]

def load_or_fetch_enrichr_library(lib_name, organism="Human", cache_dir=CACHE_DIR):
    cache_file = cache_dir / f"{lib_name}.pkl"
    if cache_file.exists():
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    gs = gp.get_library(name=lib_name, organism=organism)
    gs = canonicalize_geneset_dict(gs)

    with open(cache_file, "wb") as f:
        pickle.dump(gs, f)

    return gs

available_libs = gp.get_library_name(organism="Human")

GO_BP_NAME = latest_matching_library(available_libs, "GO_Biological_Process_")
GO_CC_NAME = latest_matching_library(available_libs, "GO_Cellular_Component_")
GO_MF_NAME = latest_matching_library(available_libs, "GO_Molecular_Function_")
KEGG_NAME = latest_matching_library(available_libs, "KEGG_")
REACTOME_NAME = latest_matching_library(available_libs, "Reactome_")

print("Using libraries:")
print("  GO BP     :", GO_BP_NAME)
print("  GO CC     :", GO_CC_NAME)
print("  GO MF     :", GO_MF_NAME)
print("  KEGG      :", KEGG_NAME)
print("  Reactome  :", REACTOME_NAME)

GO_BP = load_or_fetch_enrichr_library(GO_BP_NAME)
GO_CC = load_or_fetch_enrichr_library(GO_CC_NAME)
GO_MF = load_or_fetch_enrichr_library(GO_MF_NAME)
KEGG_PATHWAYS = load_or_fetch_enrichr_library(KEGG_NAME)
REACTOME_PATHWAYS = load_or_fetch_enrichr_library(REACTOME_NAME)

# Combine GO into one dictionary, while keeping source in the term name
GO_PATHWAYS = {}
GO_PATHWAYS.update({f"BP: {k}": v for k, v in GO_BP.items()})
GO_PATHWAYS.update({f"CC: {k}": v for k, v in GO_CC.items()})
GO_PATHWAYS.update({f"MF: {k}": v for k, v in GO_MF.items()})

print(f"PHENOTYPE_GENESETS: {len(PHENOTYPE_GENESETS)}")
print(f"GO_PATHWAYS:        {len(GO_PATHWAYS)}")
print(f"KEGG_PATHWAYS:      {len(KEGG_PATHWAYS)}")
print(f"REACTOME_PATHWAYS:  {len(REACTOME_PATHWAYS)}")

Using libraries:
  GO BP     : GO_Biological_Process_2026
  GO CC     : GO_Cellular_Component_2026
  GO MF     : GO_Molecular_Function_2026
  KEGG      : KEGG_2026
  Reactome  : Reactome_Pathways_2024


PHENOTYPE_GENESETS: 24
GO_PATHWAYS:        7852
KEGG_PATHWAYS:      352
REACTOME_PATHWAYS:  2100


## Final pathway heatmap from the original Soskic notebook

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import seaborn as sns
import gseapy as gp
import textwrap

# ============================================================
# Parameters
# ============================================================
MIN_GENESET_SIZE = 50
MAX_GENESET_SIZE = 500
MIN_RECALL = 0.0
MIN_INTERSECT = 0
TOP_N_PER_POPULATION = 5

DE_PADJ_CUTOFF = 0.05
ENRICH_FDR_CUTOFF = 0.05

# Keep all 4 rows in the heatmap
COMPARISONS = [
    {
        "key": "signal1_vs_all",
        "title": "Population 1",
        "topbar_label": "Top pathways\n for population 1",
        "topbar_color": "#1f77b4",
    },
    {
        "key": "signal2_vs_all",
        "title": "Population 2",
        "topbar_label": "Top pathways\n for population 2",
        "topbar_color": "#d62728",
    },
    {
        "key": "signal1_background_vs_all",
        "title": "All non-sig. TN NFKB",
        "topbar_label": "Top pathways for\n non-sig. TN NFKB",
        "topbar_color": "#2ca02c",
    },
    {
        "key": "signal2_background_vs_all",
        "title": "All non-sig. TN NFKB/TCM 16h/\nTEM 16h/TN 16h",
        "topbar_label": "Top pathways for\n non-sig. TN NFKB/TCM 16h/\nTEM 16h/TN 16h",
        "topbar_color": "#9467bd",
    },
]

# Use all 4 comparisons to define the column groups
COLUMN_COMPARISONS = [
    {
        "key": "signal1_vs_all",
        "title": "Population 1",
        "topbar_label": "Top pathways for\n population 1",
        "topbar_color": "#1f77b4",
    },
    {
        "key": "signal2_vs_all",
        "title": "Population 2",
        "topbar_label": "Top pathways for\n population 2",
        "topbar_color": "#d62728",
    },
    {
        "key": "signal1_background_vs_all",
        "title": "All non-sig. TN NFKB",
        "topbar_label": "Top pathways for\n non-sig. TN NFKB",
        "topbar_color": "#2ca02c",
    },
    {
        "key": "signal2_background_vs_all",
        "title": "All non-sig. TN NFKB/\nTCM 16h/TEM 16h/TN 16h",
        "topbar_label": "Top pathways for\n non-sig. TN NFKB/\nTCM 16h/TEM 16h/TN 16h",
        "topbar_color": "#9467bd",
    },
]

SAVEPATH = str(OUTPUT_DIR / "four_rows_four_column_groups_heatmap.png")   # set to None to skip saving
DPI = 300

# Fixed odds-ratio color scale
OR_VMIN = 0.1
OR_VCENTER = 1.0
OR_VMAX = 3.5

# ============================================================
# Text sizes / layout controls
# ============================================================
YTICK_FONTSIZE = 16
XTICK_FONTSIZE = 12
ANNOT_FONTSIZE = 11
CBAR_LABEL_FONTSIZE = 16
CBAR_TICK_FONTSIZE = 14
TOPBAR_LABEL_FONTSIZE = 12

ANNOT_FONTWEIGHT = "semibold"
TOPBAR_LABEL_FONTWEIGHT = "bold"

XTICK_ROTATION = 45
TERM_WRAP_WIDTH = 20

ANNOT_OUTLINE_COLOR = "black"
ANNOT_OUTLINE_WIDTH = 2.0

# ============================================================
# Label mapping dict
# ============================================================
mapping_dict = {
    "MF: Endopeptidase Regulator Activity (GO:0061135)": "Endopeptidase regulation",

    "BP: Regulation of Reactive Oxygen Species Metabolic Process (GO:2000377)": "ROS metabolism",

    "BP: Regulation of B Cell Proliferation (GO:0030888)": "B cell proliferation",

    "KEGG: P53 SIGNALING PATHWAY": "p53 signaling",

    "REACTOME: Signaling by NOTCH1": "NOTCH1 signaling",

    "BP: Protein-DNA Complex Assembly (GO:0065004)": "Protein-DNA complex assembly",

    "BP: Positive Regulation of Reactive Oxygen Species Metabolic Process (GO:2000379)": "ROS metabolism upregulation",

    "BP: Regulation of Viral Genome Replication (GO:0045069)": "Viral genome replication",

    "BP: Response to Glucose (GO:0009749)": "Glucose response",

    "BP: Regulation of Protein Kinase Activity (GO:0045859)": "Protein kinase regulation",

    "BP: Response to Peptide Hormone (GO:0043434)": "Peptide hormone response",

    "BP: Morphogenesis of an Epithelium (GO:0002009)": "Epithelial morphogenesis",

    "REACTOME: Nuclear Events (Kinase and Transcription Factor Activation)": "Nuclear signaling events",

    "REACTOME: Gap Junction Trafficking and Regulation": "Gap junction regulation",

    "REACTOME: Amyloid Fiber Formation": "Amyloid formation",

    "REACTOME: DNA Double Strand Break Response": "DSB response",

    "BP: Macromolecule Modification (GO:0043412)": "Macromolecule modification",

    "REACTOME: Recruitment and ATM-med Phosphorylation of Repair and Signaling Proteins at DNA Double Strand Breaks": "ATM repair signaling",

    "KEGG: GLYCOLYSIS / GLUCONEOGENESIS": "Glycolysis / gluconeogenesis",

    "BP: ATP Metabolic Process (GO:0046034)": "ATP metabolism",
}

# ============================================================
# All genesets
# ============================================================
ALL_GENESETS = {}
ALL_GENESETS.update({f"PHENO: {k}": v for k, v in PHENOTYPE_GENESETS.items()})
ALL_GENESETS.update({f"{k}": v for k, v in GO_PATHWAYS.items()})   # already prefixed BP:/CC:/MF:
ALL_GENESETS.update({f"KEGG: {k}": v for k, v in KEGG_PATHWAYS.items()})
ALL_GENESETS.update({f"REACTOME: {k}": v for k, v in REACTOME_PATHWAYS.items()})

ALL_GENESETS = canonicalize_geneset_dict(ALL_GENESETS)
ALL_GENESETS = {
    k: v for k, v in ALL_GENESETS.items()
    if (len(v) > MIN_GENESET_SIZE) and (len(v) < MAX_GENESET_SIZE)
}

# ============================================================
# Helpers
# ============================================================
def prep_ranked_de_table(df):
    out = df.copy()
    out["names"] = out["names"].astype(str).str.upper().str.strip()
    out = out.loc[out["names"].notna() & (out["names"] != "")]
    return out


def split_pos_neg_deg_genes(df, padj_cutoff=0.05):
    d = prep_ranked_de_table(df)
    d = d.loc[d["pvals_adj"].notna() & (d["pvals_adj"] < padj_cutoff)].copy()

    if "logfoldchanges" not in d.columns:
        raise ValueError("Expected 'logfoldchanges' column in DE table.")

    pos = d.loc[d["logfoldchanges"] > 0, "names"].drop_duplicates().tolist()
    neg = d.loc[d["logfoldchanges"] < 0, "names"].drop_duplicates().tolist()
    return pos, neg


def parse_overlap_string(x):
    """
    Expects strings like '14/123'. Returns (intersect, denominator).
    """
    if pd.isna(x):
        return (np.nan, np.nan)

    if isinstance(x, str) and "/" in x:
        a, b = x.split("/", 1)
        try:
            return (float(a), float(b))
        except ValueError:
            return (np.nan, np.nan)

    return (np.nan, np.nan)


def run_enrichr_dict(
    gene_list,
    genesets_dict,
    background_genes=None,
    enrich_fdr_cutoff=0.05,
    min_recall=0.10,
    min_intersect=10,
):
    empty_cols = [
        "Term", "Odds Ratio", "Adjusted P-value", "P-value",
        "intersect", "geneset_size", "recall", "is_significant"
    ]
    if len(gene_list) == 0:
        return pd.DataFrame(columns=empty_cols)

    enr = gp.enrich(
        gene_list=gene_list,
        gene_sets=genesets_dict,
        background=background_genes,
        outdir=None,
        no_plot=True,
        verbose=False,
    )

    if enr.results is None or len(enr.results) == 0:
        return pd.DataFrame(columns=empty_cols)

    res = enr.results.copy()

    for col in ["Adjusted P-value", "P-value", "Odds Ratio", "Combined Score"]:
        if col in res.columns:
            res[col] = pd.to_numeric(res[col], errors="coerce")

    res = res.loc[res["Term"].notna() & res["Odds Ratio"].notna()].copy()
    res = res.loc[res["Odds Ratio"] > 0].copy()

    res["geneset_size"] = res["Term"].map(lambda x: len(genesets_dict.get(x, [])))

    if "Overlap" in res.columns:
        parsed = res["Overlap"].apply(parse_overlap_string)
        res["intersect"] = parsed.apply(lambda x: x[0])
        overlap_denom = parsed.apply(lambda x: x[1])

        missing_mask = res["geneset_size"].isna()
        res.loc[missing_mask, "geneset_size"] = overlap_denom[missing_mask]
    else:
        res["intersect"] = np.nan

    res["geneset_size"] = pd.to_numeric(res["geneset_size"], errors="coerce")
    res["intersect"] = pd.to_numeric(res["intersect"], errors="coerce")
    res["recall"] = res["intersect"] / res["geneset_size"]

    res = res.loc[
        res["geneset_size"].notna()
        & (res["geneset_size"] > MIN_GENESET_SIZE)
        & (res["geneset_size"] < MAX_GENESET_SIZE)
        & res["intersect"].notna()
        & (res["intersect"] > min_intersect)
        & res["recall"].notna()
        & (res["recall"] > min_recall)
    ].copy()

    res["is_significant"] = res["Adjusted P-value"] < enrich_fdr_cutoff
    return res


def select_top_n_unique_sig_terms(target_key, enrichment_results, comparisons_for_uniqueness, top_n=10):
    """
    Take up to top_n pathways that are:
      1) significant in target_key
      2) NOT significant in any other comparison listed in comparisons_for_uniqueness
    Rank by decreasing OR, then FDR, then p-value, then name.
    """
    target_res = enrichment_results[target_key].copy()
    if target_res.empty:
        return []

    target_sig = target_res.loc[target_res["is_significant"]].copy()
    if target_sig.empty:
        return []

    other_sig_terms = set()
    for comp in comparisons_for_uniqueness:
        other_key = comp["key"]
        if other_key == target_key:
            continue

        other_res = enrichment_results[other_key]
        if other_res.empty:
            continue

        other_sig_terms.update(
            other_res.loc[other_res["is_significant"], "Term"].astype(str).tolist()
        )

    target_unique = target_sig.loc[
        ~target_sig["Term"].astype(str).isin(other_sig_terms)
    ].copy()

    if target_unique.empty:
        return []

    ranked = (
        target_unique.sort_values(
            ["Odds Ratio", "Adjusted P-value", "P-value", "Term"],
            ascending=[False, True, True, True]
        )["Term"]
        .head(top_n)
        .tolist()
    )
    return ranked


def build_grouped_topn_table_multi(enrichment_results, all_comparisons, column_comparisons, top_n=10):
    """
    Build full merged table using all comparisons for rows.
    Only create column blocks for column_comparisons.
    """
    merged = None

    for comp in all_comparisons:
        key = comp["key"]
        res = enrichment_results[key][[
            "Term", "Odds Ratio", "Adjusted P-value", "is_significant",
            "intersect", "geneset_size", "recall"
        ]].copy().rename(columns={
            "Odds Ratio": f"{key}_or",
            "Adjusted P-value": f"{key}_fdr",
            "is_significant": f"{key}_sig",
            "intersect": f"{key}_intersect",
            "geneset_size": f"{key}_geneset_size",
            "recall": f"{key}_recall",
        })

        merged = res if merged is None else merged.merge(res, on="Term", how="outer")

    if merged is None or merged.empty:
        return pd.DataFrame()

    for comp in all_comparisons:
        key = comp["key"]
        merged[f"{key}_or"] = merged[f"{key}_or"].fillna(1.0)
        merged[f"{key}_fdr"] = merged[f"{key}_fdr"].fillna(1.0)
        merged[f"{key}_sig"] = merged[f"{key}_sig"].fillna(False)

    blocks = []
    for comp in column_comparisons:
        key = comp["key"]

        top_terms = select_top_n_unique_sig_terms(
            target_key=key,
            enrichment_results=enrichment_results,
            comparisons_for_uniqueness=all_comparisons,
            top_n=top_n,
        )

        if len(top_terms) == 0:
            continue

        block = merged.loc[merged["Term"].isin(top_terms)].copy()
        block["panel_rank"] = block["Term"].map({t: i for i, t in enumerate(top_terms)})
        block["panel"] = key
        block["panel_label"] = comp["topbar_label"]
        block["panel_color"] = comp["topbar_color"]
        block = block.sort_values("panel_rank").copy()
        blocks.append(block)

    if len(blocks) == 0:
        return pd.DataFrame()

    plot_df = pd.concat(blocks, axis=0, ignore_index=True)
    return plot_df


def pretty_term(term, width=18, mapping_dict=None):
    original = str(term).strip()

    if mapping_dict is not None and original in mapping_dict:
        label = mapping_dict[original]
    else:
        label = original
        label = label.replace("PHENO: ", "")
        label = label.replace("KEGG: ", "KEGG ")
        label = label.replace("REACTOME: ", "Reactome ")
        label = label.replace("BP: ", "GO-BP ")
        label = label.replace("CC: ", "GO-CC ")
        label = label.replace("MF: ", "GO-MF ")
        label = label.replace("_", " ")

    return "\n".join(textwrap.wrap(label, width=width))


def draw_outlined_text(ax, x, y, text, fontsize, fontweight="normal",
                       color="white", outline_color="black", outline_width=2.0):
    t = ax.text(
        x, y, text,
        ha="center", va="center",
        fontsize=fontsize,
        color=color,
        fontweight=fontweight,
    )
    t.set_path_effects([
        pe.Stroke(linewidth=outline_width, foreground=outline_color),
        pe.Normal()
    ])
    return t


def plot_grouped_multi_heatmap(df, row_comparisons, column_comparisons, norm, cmap="RdBu_r", mapping_dict=None):
    if df.empty:
        raise ValueError("No pathways available after filtering.")

    values = np.vstack([
        df[f"{comp['key']}_or"].to_numpy(dtype=float)
        for comp in row_comparisons
    ])

    sigs = np.vstack([
        df[f"{comp['key']}_sig"].astype(bool).to_numpy()
        for comp in row_comparisons
    ])

    xlabels = [pretty_term(x, width=TERM_WRAP_WIDTH, mapping_dict=mapping_dict) for x in df["Term"]]

    ncols = len(df)
    nrows = len(row_comparisons)

    fig_w = 15
    fig_h = 5
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(values, aspect="auto", cmap=cmap, norm=norm)

    ax.set_xticks(np.arange(ncols))
    ax.set_xticklabels(xlabels, rotation=XTICK_ROTATION, ha="right", fontsize=XTICK_FONTSIZE)

    ax.set_yticks(np.arange(nrows))
    ax.set_yticklabels([comp["title"] for comp in row_comparisons], fontsize=YTICK_FONTSIZE)

    ax.set_xticks(np.arange(-0.5, ncols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nrows, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    for sp in ax.spines.values():
        sp.set_visible(False)

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if sigs[i, j]:
                txt = f"{values[i, j]:.1f}"
                draw_outlined_text(
                    ax,
                    x=j,
                    y=i,
                    text=txt,
                    fontsize=ANNOT_FONTSIZE,
                    fontweight=ANNOT_FONTWEIGHT,
                    color="white",
                    outline_color=ANNOT_OUTLINE_COLOR,
                    outline_width=ANNOT_OUTLINE_WIDTH,
                )

    ax.set_xlim(-0.5, ncols - 0.5)
    ax.set_ylim(nrows - 0.5, -1.35)

    bar_y = -0.96
    label_y = -1.45

    start_idx = 0
    for idx, comp in enumerate(column_comparisons):
        block_size = int((df["panel"] == comp["key"]).sum())
        if block_size == 0:
            continue

        x0 = start_idx - 0.5
        x1 = start_idx + block_size - 0.5

        ax.plot(
            [x0, x1],
            [bar_y, bar_y],
            color=comp["topbar_color"],
            linewidth=6,
            solid_capstyle="butt",
            clip_on=False
        )

        ax.text(
            start_idx + (block_size - 1) / 2,
            label_y - 0.1,
            comp["topbar_label"],
            ha="center",
            va="center",
            fontsize=TOPBAR_LABEL_FONTSIZE,
            fontweight=TOPBAR_LABEL_FONTWEIGHT,
            color=comp["topbar_color"],
            clip_on=False
        )

        start_idx += block_size

        if idx < (len(column_comparisons) - 1):
            ax.vlines(
                x=start_idx - 0.5,
                ymin=-0.5,
                ymax=nrows - 0.5,
                color="black",
                linewidth=1.5
            )

    # Vertical colorbar on the right
    cbar = fig.colorbar(im, ax=ax, orientation="vertical", pad=0.02, fraction=0.04)
    cbar.set_label("Odds ratio", fontsize=CBAR_LABEL_FONTSIZE)
    cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE)

    plt.tight_layout()

    if SAVEPATH is not None:
        plt.savefig(SAVEPATH, dpi=DPI, bbox_inches="tight")

    plt.show()


# ============================================================
# Background genes for ORA
# Use the union of all genes seen across the 4 DE tables
# ============================================================
background_gene_sets = []
for comp in COMPARISONS:
    key = comp["key"]
    ranked = prep_ranked_de_table(deg_results[key]["all_ranked_genes"])
    background_gene_sets.append(set(ranked["names"]))

BACKGROUND_GENES = sorted(set().union(*background_gene_sets))


# ============================================================
# Positive DE genes only for each of the 4 populations
# ============================================================
pos_deg_genes = {}
enrichment_results = {}

for comp in COMPARISONS:
    key = comp["key"]
    de_table = deg_results[key]["all_ranked_genes"].copy()

    pos, _ = split_pos_neg_deg_genes(de_table, padj_cutoff=DE_PADJ_CUTOFF)
    pos_deg_genes[key] = pos

    enrichment_results[key] = run_enrichr_dict(
        pos,
        ALL_GENESETS,
        background_genes=BACKGROUND_GENES,
        enrich_fdr_cutoff=ENRICH_FDR_CUTOFF,
        min_recall=MIN_RECALL,
        min_intersect=MIN_INTERSECT,
    )

# Add the positive-DE gene-list size used for pathway enrichment to each
# heatmap row label, e.g. "Population 1 (23 genes)".
for comp in COMPARISONS:
    n_genes = len(pos_deg_genes[comp["key"]])
    comp["title"] = f'{comp["title"]} ({n_genes:,} genes)'


# ============================================================
# Build grouped plot table:
# - rows are all 4 comparisons
# - columns now also include both background comparison groups
# ============================================================
plot_df = build_grouped_topn_table_multi(
    enrichment_results,
    all_comparisons=COMPARISONS,
    column_comparisons=COLUMN_COMPARISONS,
    top_n=TOP_N_PER_POPULATION,
)

if plot_df.empty:
    raise ValueError("No uniquely significant enriched pathways found for the selected column groups.")

display(plot_df[["Term", "panel"]].head(40))


# ============================================================
# Plot
# ============================================================
sns.set_style("white")
norm = mcolors.TwoSlopeNorm(vmin=OR_VMIN, vcenter=OR_VCENTER, vmax=OR_VMAX)

plot_grouped_multi_heatmap(
    plot_df,
    row_comparisons=COMPARISONS,
    column_comparisons=COLUMN_COMPARISONS,
    norm=norm,
    cmap="RdBu_r",
    mapping_dict=mapping_dict,
)

,Term,panel
0,BP: Antibacterial Humoral Response (GO:0019731),signal1_background_vs_all
1,BP: Natural Killer Cell Activation (GO:0030101),signal1_background_vs_all
2,REACTOME: Interferon Alpha Beta Signaling,signal1_background_vs_all
3,BP: Cellular Response to Virus (GO:0098586),signal1_background_vs_all
4,BP: Regulation of Smooth Muscle Cell Prolifera...,signal1_background_vs_all
5,REACTOME: tRNA Processing in the Nucleus,signal2_background_vs_all
6,REACTOME: HCMV Late Events,signal2_background_vs_all
7,BP: Nucleus Organization (GO:0006997),signal2_background_vs_all
8,REACTOME: HCMV Infection,signal2_background_vs_all
9,REACTOME: Signaling by FGFR in Disease,signal2_background_vs_all


In [25]:
list(plot_df['Term'].unique())

['BP: Antibacterial Humoral Response (GO:0019731)',
 'BP: Natural Killer Cell Activation (GO:0030101)',
 'REACTOME: Interferon Alpha Beta Signaling',
 'BP: Cellular Response to Virus (GO:0098586)',
 'BP: Regulation of Smooth Muscle Cell Proliferation (GO:0048660)',
 'REACTOME: tRNA Processing in the Nucleus',
 'REACTOME: HCMV Late Events',
 'BP: Nucleus Organization (GO:0006997)',
 'REACTOME: HCMV Infection',
 'REACTOME: Signaling by FGFR in Disease']

## Optional: Soskic cell-population counts

In [26]:
celltype_counts(soskic_adata, soskic_config.biocol)

Cell_population
TN 0h               1633
TN 40h               986
TN 16h               803
TCM 0h               640
TCM 5d               614
TCM 16h              591
TN 5d                446
TN HSP 5d            364
TCM 40h              336
TEM 0h               328
TEM 5d               302
TN IFN 5d            291
TEM 16h              283
TEM 40h              270
TN LA                259
TCM LA               236
TN IFN 40h           195
TEM LA               186
TN cycling 40h       130
TN NFKB              106
TN cycling 5d         98
T ER-stress 5d        93
TM ER-stress 40h      91
TN IFN LA             80
nTreg 40h             79
nTreg 16h             79
TEMRA 0h              59
TEM HLA+ 5d           58
TEM HLA+ 40h          56
TEMRA 16h             48
TEMRA 40h             44
TEMRA 5d              44
TM cycling 5d         37
nTreg 0h              34
TN2 40h               27
TN IFN 16h            25
HSP 16h               25
TEMRA LA              24
Name: count, dtype: int64